# CHW Program — Databricks Lakehouse Semantic Model
**Author:** Erick Kiprotich Yegon, PhD  
**Dataset:** Kenya Community Health Worker (CHW) Program — 12 CSV sources  
**Architecture:** Medallion (Bronze → Silver → Gold) + Unity Catalog Semantic Layer  
**Consumer:** Power BI DirectQuery + Databricks AI/BI Genie

---
## Notebook structure
| Section | Layer | Description |
|---------|-------|-------------|
| 00 | Setup | Unity Catalog, schemas, volumes |
| 01 | Bronze | Raw CSV ingestion via Auto Loader |
| 02 | Silver | Cleaning, typing, dedup, merging |
| 03 | Gold | Star schema — facts & dimensions |
| 04 | Semantic | Metric views, semantic views, RLS |
| 05 | Validate | Data quality checks per layer |
| 06 | BI | Power BI connection guide + DAX |

## 00 · Setup — Unity Catalog, Schemas & Volumes

In [0]:
# ── 00.1  Unity Catalog setup ──────────────────────────────────────────────
# Run once per workspace. Requires CATALOG CREATE privilege.

spark.sql("""
    CREATE CATALOG IF NOT EXISTS chw_catalog
    COMMENT 'Kenya CHW program — all medallion layers'
""")

for schema, desc in [
    ('bronze', 'Raw ingested CSVs — no transforms'),
    ('silver', 'Cleaned, typed, deduplicated tables'),
    ('gold',   'Star schema for BI consumption'),
]:
    spark.sql(f"""
        CREATE SCHEMA IF NOT EXISTS chw_catalog.{schema}
        COMMENT '{desc}'
    """)

spark.sql("""
    CREATE VOLUME IF NOT EXISTS chw_catalog.bronze.uploads
    COMMENT 'Landing zone for source CSV files'
""")
print('✓ Catalog, schemas, and volume created.')

✓ Catalog, schemas, and volume created.


In [0]:
# ── 00.2  Shared config ────────────────────────────────────────────────────

CATALOG      = 'chw_catalog'
UPLOAD_PATH  = f'/Volumes/{CATALOG}/bronze/uploads'
CHECKPOINT   = f'/Volumes/{CATALOG}/bronze/checkpoints'
SCHEMA_HINTS = f'/Volumes/{CATALOG}/bronze/schema_hints'

SOURCE_TABLES = [
    'iz', 'supervision',
    'preg_reg', 'preg_visit',
    'pnc', 'fp', 'refill',
    'homevisit', 'population',
    'active_chps', 'household',
]

spark.sql(f'USE CATALOG {CATALOG}')
print(f'✓ Working in catalog: {CATALOG}')

✓ Working in catalog: chw_catalog


## 01 · Bronze — Raw CSV Ingestion (Auto Loader)

In [0]:
# ── 01  Bronze ingestion via COPY INTO (Serverless compatible) ─────────────

for table_name in SOURCE_TABLES:
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS chw_catalog.bronze.{table_name}
        USING DELTA
        TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'true')
    """)
    spark.sql(f"""
        COPY INTO chw_catalog.bronze.{table_name}
        FROM '/Volumes/chw_catalog/bronze/uploads/{table_name}.csv'
        FILEFORMAT = CSV
        FORMAT_OPTIONS (
            'header'      = 'true',
            'inferSchema' = 'true',
            'multiLine'   = 'true',
            'escape'      = '"'
        )
        COPY_OPTIONS (
            'force'       = 'true',
            'mergeSchema' = 'true'
        )
    """)
    print(f'  ✓ {table_name}')

print('\n✓ All Bronze tables loaded.')

  ✓ iz
  ✓ supervision
  ✓ preg_reg
  ✓ preg_visit
  ✓ pnc
  ✓ fp
  ✓ refill
  ✓ homevisit
  ✓ population
  ✓ active_chps
  ✓ household

✓ All Bronze tables loaded.


In [0]:
# ── 01.2  Alternative: COPY INTO (batch, no streaming cluster needed) ──────

for table_name in SOURCE_TABLES:
    spark.sql(f'''
        CREATE TABLE IF NOT EXISTS {CATALOG}.bronze.{table_name}
        USING DELTA
        TBLPROPERTIES ("delta.autoOptimize.optimizeWrite" = "true")
    ''')
    spark.sql(f'''
        COPY INTO {CATALOG}.bronze.{table_name}
        FROM "{UPLOAD_PATH}/{table_name}.csv"
        FILEFORMAT = CSV
        FORMAT_OPTIONS (
            "header"      = "true",
            "inferSchema" = "true",
            "multiLine"   = "true",
            "escape"      = "\\"")
        COPY_OPTIONS ("force" = "false")
    ''')
    print(f'  ✓ COPY INTO bronze.{table_name}')

  ✓ COPY INTO bronze.iz
  ✓ COPY INTO bronze.supervision
  ✓ COPY INTO bronze.preg_reg
  ✓ COPY INTO bronze.preg_visit
  ✓ COPY INTO bronze.pnc
  ✓ COPY INTO bronze.fp
  ✓ COPY INTO bronze.refill
  ✓ COPY INTO bronze.homevisit
  ✓ COPY INTO bronze.population
  ✓ COPY INTO bronze.active_chps
  ✓ COPY INTO bronze.household


## 02 · Silver — Clean, Type, Deduplicate & Merge

In [0]:
# ── 02.1  silver.active_chps ───────────────────────────────────────────────

spark.sql('''
CREATE OR REPLACE TABLE chw_catalog.silver.active_chps AS
SELECT
  chw_uuid,
  chw_area_uuid,
  chw_area_name,
  chw_name,
  chw_phone,
  cha_name,
  community_unit,
  county                         AS county_code,
  county_name,
  sub_county_name,
  CAST(reported AS TIMESTAMP)    AS activated_at,
  reportedm                      AS report_month,
  current_timestamp()            AS silver_loaded_at
FROM chw_catalog.bronze.active_chps
WHERE chw_uuid IS NOT NULL
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY chw_uuid ORDER BY reported DESC
) = 1
''')
print('✓ silver.active_chps')

✓ silver.active_chps


In [0]:
# ── 02.2  silver.immunization ──────────────────────────────────────────────

spark.sql('''
CREATE OR REPLACE TABLE chw_catalog.silver.immunization AS
SELECT
  uuid,
  CAST(reported AS TIMESTAMP)                 AS reported_at,
  county, month, reportedm,
  patient_id, patient_name,
  patient_sex                                 AS sex,
  CAST(patient_age_in_months AS INT)          AS age_months,
  CAST(patient_age_in_years AS INT)           AS age_years,
  contact_parent_id                           AS chw_area_uuid,
  chw_name,
  CAST(COALESCE(has_good_immunization_status = "yes", false) AS INT) AS is_fully_immunized,
  CAST(COALESCE(needs_follow_up = "yes", false) AS INT)              AS is_defaulter,
  CAST(COALESCE(has_been_referred = "yes", false) AS INT)            AS is_referred,
  COALESCE(CAST(has_bcg AS INT), 0)           AS bcg,
  COALESCE(CAST(has_opv_0 AS INT), 0)         AS opv0,
  COALESCE(CAST(has_opv_1 AS INT), 0)         AS opv1,
  COALESCE(CAST(has_opv_2 AS INT), 0)         AS opv2,
  COALESCE(CAST(has_opv_3 AS INT), 0)         AS opv3,
  COALESCE(CAST(has_penta_1 AS INT), 0)       AS penta1,
  COALESCE(CAST(has_penta_2 AS INT), 0)       AS penta2,
  COALESCE(CAST(has_penta_3 AS INT), 0)       AS penta3,
  COALESCE(CAST(has_pcv_1 AS INT), 0)         AS pcv1,
  COALESCE(CAST(has_pcv_3 AS INT), 0)         AS pcv3,
  COALESCE(CAST(has_measles_9_months AS INT), 0)  AS measles_9m,
  COALESCE(CAST(has_measles_18_months AS INT), 0) AS measles_18m,
  COALESCE(CAST(has_vitamin_a AS INT), 0)     AS vitamin_a,
  COALESCE(CAST(total_due_vaccines AS INT), 0) AS vaccines_due,
  CAST(latitude AS DOUBLE)                    AS lat,
  CAST(longitude AS DOUBLE)                   AS lon,
  referred_to_facility_code,
  referred_to_facility_name,
  current_timestamp()                         AS silver_loaded_at
FROM chw_catalog.bronze.iz
WHERE uuid IS NOT NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY uuid ORDER BY reported DESC) = 1
''')
print('✓ silver.immunization')

✓ silver.immunization


In [0]:
# ── 02.3  silver.pregnancy_registration ────────────────────────────────────

spark.sql('''
CREATE OR REPLACE TABLE chw_catalog.silver.pregnancy_registration AS
SELECT
    uuid,
    CAST(reported AS TIMESTAMP)                                            AS reported_at,
    month, reportedm, patient_id, patient_name,
    TRY_CAST(patient_age_in_years AS INT)                                  AS age_years,
    CAST(COALESCE(has_started_anc = "yes", false) AS INT)                  AS has_started_anc,
    COALESCE(TRY_CAST(number_of_anc_attended AS INT), 0)                   AS anc_visits,
    TRY_CAST(next_anc_visit_date AS DATE)                                  AS next_anc_date,
    TRY_CAST(updated_edd AS DATE)                                          AS edd,
    CAST(COALESCE(has_danger_signs = "yes", false) AS INT)                 AS has_danger_signs,
    muac_color,
    CAST(COALESCE(takes_iron_or_folate_supplements = "yes", false) AS INT) AS on_iron_folate,
    CAST(COALESCE(TRY_CAST(is_anc_defaulter AS BOOLEAN), false) AS INT)    AS is_anc_defaulter,
    CAST(COALESCE(has_been_referred = "yes", false) AS INT)                AS is_referred,
    chu_code, chu_name,
    referred_to_facility_code, referred_to_facility_name,
    "preg_reg"                                                             AS source_form,
    current_timestamp()                                                    AS silver_loaded_at
FROM chw_catalog.bronze.preg_reg
WHERE uuid IS NOT NULL
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY patient_id, reportedm ORDER BY reported DESC
) = 1
''')
print('✓ silver.pregnancy_registration')

✓ silver.pregnancy_registration


In [0]:
# ── 02.4–02.7  Remaining Silver tables ─────────────────────────────────────

# postnatal_care
spark.sql('''
CREATE OR REPLACE TABLE chw_catalog.silver.postnatal_care AS
SELECT
  uuid,
  CAST(reported AS TIMESTAMP)                                               AS reported_at,
  month, reportedm,
  patient_id, patient_name, patient_sex                                     AS sex,
  TRY_CAST(patient_age_in_years AS INT)                                     AS age_years,
  contact_parent_id                                                         AS chw_area_uuid,
  TRY_CAST(date_of_delivery AS DATE)                                        AS delivery_date,
  place_of_delivery,
  TRY_CAST(days_since_delivery AS INT)                                      AS days_since_delivery,
  TRY_CAST(postnatal_care_service_count AS INT)                             AS pnc_visit_count,
  CAST(COALESCE(has_pnc_up_to_date = "yes", false) AS INT)                  AS pnc_upto_date,
  condition_of_the_mother,
  CAST(COALESCE(TRY_CAST(needs_danger_signs_follow_up AS BOOLEAN), false) AS INT) AS needs_danger_followup,
  CAST(COALESCE(TRY_CAST(is_maternal_death AS BOOLEAN), false) AS INT)      AS is_maternal_death,
  CAST(COALESCE(has_started_family_planning = "yes", false) AS INT)         AS started_fp,
  resident_chu_code, resident_chu_name,
  TRY_CAST(newborn_home_visit_count AS INT)                                 AS newborn_visits,
  TRY_CAST(latitude AS DOUBLE)                                              AS lat,
  TRY_CAST(longitude AS DOUBLE)                                             AS lon,
  current_timestamp()                                                       AS silver_loaded_at
FROM chw_catalog.bronze.pnc
WHERE uuid IS NOT NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY uuid ORDER BY reported DESC) = 1
''')
print('✓ silver.postnatal_care')

# family_planning (fp + refill merged)
spark.sql('''
CREATE OR REPLACE TABLE chw_catalog.silver.family_planning AS
WITH base AS (
  SELECT
    uuid, CAST(reported AS TIMESTAMP)                                       AS reported_at,
    month, reportedm, patient_id, patient_name, patient_sex                 AS sex,
    TRY_CAST(patient_age_in_years AS INT)                                   AS age_years,
    contact_parent_id                                                       AS chw_area_uuid,
    CAST(COALESCE(on_fp = "yes", false) AS INT)                             AS is_on_fp,
    coalesced_fp_method                                                     AS fp_method,
    CAST(COALESCE(refilled_today = "yes", false) AS INT)                    AS refilled_today,
    COALESCE(TRY_CAST(coc_cycles AS INT), 0)                                AS coc_cycles,
    COALESCE(TRY_CAST(pop_cycles AS INT), 0)                                AS pop_cycles,
    COALESCE(TRY_CAST(condom_pieces AS INT), 0)                             AS condom_pieces,
    CAST(COALESCE(has_side_effects = "yes", false) AS INT)                  AS has_side_effects,
    CAST(COALESCE(is_referral = "yes", false) AS INT)                       AS is_referral,
    referred_to_facility_code, referred_to_facility_name,
    TRY_CAST(latitude AS DOUBLE)                                            AS lat,
    TRY_CAST(longitude AS DOUBLE)                                           AS lon,
    "fp"                                                                    AS source_form
  FROM chw_catalog.bronze.fp WHERE uuid IS NOT NULL

  UNION ALL

  SELECT
    uuid, CAST(reported AS TIMESTAMP),
    month, reportedm, patient_id, patient_name, patient_sex,
    TRY_CAST(patient_age_in_years AS INT),
    contact_parent_id,
    CAST(COALESCE(on_fp = "yes", false) AS INT),
    coalesced_fp_method,
    CAST(COALESCE(refilled_today = "yes", false) AS INT),
    COALESCE(TRY_CAST(coc_cycles AS INT), 0),
    COALESCE(TRY_CAST(pop_cycles AS INT), 0),
    COALESCE(TRY_CAST(condom_pieces AS INT), 0),
    CAST(COALESCE(has_side_effects = "yes", false) AS INT),
    CAST(COALESCE(is_referral = "yes", false) AS INT),
    referred_to_facility_code, referred_to_facility_name,
    TRY_CAST(latitude AS DOUBLE),
    TRY_CAST(longitude AS DOUBLE),
    "refill"                                                                AS source_form
  FROM chw_catalog.bronze.refill WHERE uuid IS NOT NULL
)
SELECT *, current_timestamp() AS silver_loaded_at FROM base
QUALIFY ROW_NUMBER() OVER (PARTITION BY uuid ORDER BY reported_at DESC) = 1
''')
print('✓ silver.family_planning')

# supervision
spark.sql('''
CREATE OR REPLACE TABLE chw_catalog.silver.supervision AS
SELECT
  uuid,
  CAST(reported AS TIMESTAMP)                                               AS reported_at,
  month, reportedm,
  chw_uuid, chv_name                                                        AS supervisor_name,
  TRY_CAST(last_visit_date AS DATE)                                         AS last_visit_date,
  TRY_CAST(supervision_visit_count AS INT)                                  AS visit_count,
  ROUND(CASE WHEN calc_assessment_denominator > 0
    THEN calc_assessment_score / calc_assessment_denominator * 100
    ELSE NULL END, 1)                                                       AS overall_score_pct,
  ROUND(CASE WHEN calc_immunization_denominator > 0
    THEN calc_immunization_score / calc_immunization_denominator * 100
    ELSE NULL END, 1)                                                       AS immunization_score_pct,
  ROUND(CASE WHEN calc_pregnancy_home_visit_denominator > 0
    THEN calc_pregnancy_home_visit_score / calc_pregnancy_home_visit_denominator * 100
    ELSE NULL END, 1)                                                       AS pregnancy_score_pct,
  ROUND(CASE WHEN calc_nutrition_denominator > 0
    THEN calc_nutrition_score / calc_nutrition_denominator * 100
    ELSE NULL END, 1)                                                       AS nutrition_score_pct,
  ROUND(CASE WHEN calc_malaria_denominator > 0
    THEN calc_malaria_score / calc_malaria_denominator * 100
    ELSE NULL END, 1)                                                       AS malaria_score_pct,
  ROUND(CASE WHEN calc_family_planning_denominator > 0
    THEN calc_family_planning_score / calc_family_planning_denominator * 100
    ELSE NULL END, 1)                                                       AS fp_score_pct,
  ROUND(CASE WHEN calc_newborn_visit_denominator > 0
    THEN calc_newborn_visit_score / calc_newborn_visit_denominator * 100
    ELSE NULL END, 1)                                                       AS newborn_score_pct,
  ROUND(CASE WHEN calc_wash_denominator > 0
    THEN calc_wash_score / calc_wash_denominator * 100
    ELSE NULL END, 1)                                                       AS wash_score_pct,
  CAST(COALESCE(TRY_CAST(calc_has_all_tools AS BOOLEAN), false) AS INT)     AS has_all_tools,
  CAST(COALESCE(TRY_CAST(calc_has_proper_protective_equipment AS BOOLEAN), false) AS INT) AS has_ppe,
  current_timestamp()                                                       AS silver_loaded_at
FROM chw_catalog.bronze.supervision
WHERE uuid IS NOT NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY uuid ORDER BY reported DESC) = 1
''')
print('✓ silver.supervision')

# population
spark.sql('''
CREATE OR REPLACE TABLE chw_catalog.silver.population AS
SELECT
  reportedm                                                                 AS report_month,
  chw_area                                                                  AS chw_area_uuid,
  chw_uuid,
  COALESCE(TRY_CAST(u2_pop AS INT), 0)                                      AS u2_pop,
  COALESCE(TRY_CAST(u5_pop AS INT), 0)                                      AS u5_pop,
  COALESCE(TRY_CAST(wra_pop AS INT), 0)                                     AS wra_pop,
  current_timestamp()                                                       AS silver_loaded_at
FROM chw_catalog.bronze.population
WHERE chw_area IS NOT NULL
''')
print('✓ silver.population')

# home_visit
spark.sql('''
CREATE OR REPLACE TABLE chw_catalog.silver.home_visit AS
SELECT
  record_hash                                                               AS visit_key,
  month, reportedm,
  family_id,
  chw_area                                                                  AS chw_area_uuid,
  chw_uuid,
  CAST(reported AS TIMESTAMP)                                               AS visited_at,
  current_timestamp()                                                       AS silver_loaded_at
FROM chw_catalog.bronze.homevisit
WHERE record_hash IS NOT NULL
''')
print('✓ silver.home_visit')

✓ silver.postnatal_care
✓ silver.family_planning
✓ silver.supervision
✓ silver.population
✓ silver.home_visit


## 03 · Gold — Star Schema (Facts & Dimensions)

In [0]:
# ── 03.1  Dimension tables ─────────────────────────────────────────────────

# dim_chw
spark.sql('''
CREATE OR REPLACE TABLE chw_catalog.gold.dim_chw AS
SELECT DISTINCT
  chw_uuid, chw_area_uuid, chw_area_name,
  chw_name, chw_phone, cha_name, community_unit,
  county_name, sub_county_name
FROM chw_catalog.silver.active_chps
''')
print('✓ gold.dim_chw')

# dim_date
spark.sql('''
CREATE OR REPLACE TABLE chw_catalog.gold.dim_date AS
SELECT
  CAST(DATE_FORMAT(d, "yyyyMM") AS INT)   AS date_key,
  d                                        AS month_start,
  DATE_FORMAT(d, "MMM yy")               AS month_label,
  MONTH(d)                                 AS month_num,
  QUARTER(d)                               AS quarter,
  YEAR(d)                                  AS year,
  CONCAT("Q", QUARTER(d), " ", YEAR(d))  AS quarter_label
FROM (
  SELECT EXPLODE(
    SEQUENCE(DATE"2024-12-01", DATE"2026-12-01", INTERVAL 1 MONTH)
  ) AS d
)
''')
print('✓ gold.dim_date')

# dim_geography
spark.sql('''
CREATE OR REPLACE TABLE chw_catalog.gold.dim_geography AS
SELECT
  CONCAT(COALESCE(county_name, "Unknown"), "_",
         COALESCE(sub_county_name, ""))   AS geo_key,
  county_name,
  sub_county_name,
  community_unit
FROM chw_catalog.silver.active_chps
GROUP BY ALL
''')
print('✓ gold.dim_geography')

# dim_facility — from immunization, pregnancy_registration, family_planning only
spark.sql('''
CREATE OR REPLACE TABLE chw_catalog.gold.dim_facility AS
SELECT DISTINCT
  CAST(referred_to_facility_code AS STRING) AS facility_code,
  referred_to_facility_name                  AS facility_name
FROM (
  SELECT referred_to_facility_code, referred_to_facility_name
  FROM chw_catalog.silver.immunization
  WHERE referred_to_facility_code IS NOT NULL
  UNION
  SELECT referred_to_facility_code, referred_to_facility_name
  FROM chw_catalog.silver.pregnancy_registration
  WHERE referred_to_facility_code IS NOT NULL
  UNION
  SELECT referred_to_facility_code, referred_to_facility_name
  FROM chw_catalog.silver.family_planning
  WHERE referred_to_facility_code IS NOT NULL
)
''')
print('✓ gold.dim_facility')

# dim_patient
spark.sql('''
CREATE OR REPLACE TABLE chw_catalog.gold.dim_patient AS
WITH all_patients AS (
  SELECT patient_id, patient_name, sex, age_years
  FROM chw_catalog.silver.immunization
  UNION
  SELECT patient_id, patient_name, NULL AS sex, age_years
  FROM chw_catalog.silver.pregnancy_registration
  UNION
  SELECT patient_id, patient_name, sex, age_years
  FROM chw_catalog.silver.family_planning
)
SELECT patient_id, patient_name, sex, age_years
FROM all_patients
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY patient_id ORDER BY age_years DESC NULLS LAST
) = 1
''')
print('✓ gold.dim_patient')

✓ gold.dim_chw
✓ gold.dim_date
✓ gold.dim_geography
✓ gold.dim_facility
✓ gold.dim_patient


In [0]:
# fact_family_planning — rerun only this block
spark.sql('''
CREATE OR REPLACE TABLE chw_catalog.gold.fact_family_planning AS
SELECT
  fp.uuid                                                     AS fact_key,
  CAST(DATE_FORMAT(fp.reported_at, "yyyyMM") AS INT)          AS date_key,
  COALESCE(c.chw_uuid, fp.chw_area_uuid)                      AS chw_uuid,
  fp.reportedm, fp.patient_id, fp.age_years, fp.sex,
  fp.is_on_fp, fp.fp_method, fp.refilled_today,
  fp.coc_cycles, fp.pop_cycles, fp.condom_pieces,
  fp.has_side_effects, fp.is_referral, fp.source_form,
  fp.lat, fp.lon,
  CASE
    WHEN LOWER(COALESCE(fp.fp_method,"")) LIKE "%condom%"   THEN "Barrier"
    WHEN LOWER(COALESCE(fp.fp_method,"")) LIKE "%coc%"
      OR LOWER(COALESCE(fp.fp_method,"")) LIKE "%pill%"     THEN "Short-acting hormonal"
    WHEN LOWER(COALESCE(fp.fp_method,"")) LIKE "%inject%"   THEN "Injectable"
    WHEN LOWER(COALESCE(fp.fp_method,"")) LIKE "%implant%"
      OR LOWER(COALESCE(fp.fp_method,"")) LIKE "%iud%"      THEN "Long-acting"
    ELSE "Other" END                                          AS fp_method_category
FROM chw_catalog.silver.family_planning fp
LEFT JOIN chw_catalog.gold.dim_chw c
       ON fp.chw_area_uuid = c.chw_area_uuid
''')
print('✓ gold.fact_family_planning')

# fact_supervision
spark.sql('''
CREATE OR REPLACE TABLE chw_catalog.gold.fact_supervision AS
SELECT
  s.uuid                                                      AS fact_key,
  CAST(DATE_FORMAT(s.reported_at, "yyyyMM") AS INT)           AS date_key,
  s.chw_uuid, s.reportedm,
  s.supervisor_name, s.last_visit_date, s.visit_count,
  s.overall_score_pct,
  s.immunization_score_pct, s.pregnancy_score_pct,
  s.nutrition_score_pct,   s.malaria_score_pct,
  s.fp_score_pct,          s.newborn_score_pct,
  s.wash_score_pct,
  s.has_all_tools, s.has_ppe
FROM chw_catalog.silver.supervision s
''')
print('✓ gold.fact_supervision')

# fact_population
spark.sql('''
CREATE OR REPLACE TABLE chw_catalog.gold.fact_population AS
SELECT
  p.report_month,
  p.chw_uuid,
  p.chw_area_uuid,
  COALESCE(p.u2_pop, 0)  AS u2_pop,
  COALESCE(p.u5_pop, 0)  AS u5_pop,
  COALESCE(p.wra_pop, 0) AS wra_pop
FROM chw_catalog.silver.population p
''')
print('✓ gold.fact_population')

# fact_home_visit
spark.sql('''
CREATE OR REPLACE TABLE chw_catalog.gold.fact_home_visit AS
SELECT
  h.visit_key                                                 AS fact_key,
  CAST(DATE_FORMAT(h.visited_at, "yyyyMM") AS INT)            AS date_key,
  COALESCE(c.chw_uuid, h.chw_uuid)                            AS chw_uuid,
  h.reportedm, h.family_id,
  1                                                           AS visit_count
FROM chw_catalog.silver.home_visit h
LEFT JOIN chw_catalog.gold.dim_chw c
       ON h.chw_uuid = c.chw_uuid
''')
print('✓ gold.fact_home_visit')

print('\n✓ All Gold tables created.')

✓ gold.fact_family_planning
✓ gold.fact_supervision
✓ gold.fact_population
✓ gold.fact_home_visit

✓ All Gold tables created.


## 04 · Semantic Model — Views, Metrics & Row-Level Security

In [0]:
# ── 04.1  Semantic views (Power BI consumption layer) ─────────────────────

# Executive summary
spark.sql('''
CREATE OR REPLACE VIEW chw_catalog.gold.vw_executive_summary AS
WITH imm AS (
  SELECT date_key, chw_uuid,
    COUNT(*)                AS iz_visits,
    SUM(is_fully_immunized) AS iz_fully_immunized,
    SUM(is_defaulter)       AS iz_defaulters
  FROM chw_catalog.gold.fact_immunization GROUP BY ALL
),
preg AS (
  SELECT date_key, chu_code,
    COUNT(*)                AS preg_registrations,
    SUM(is_anc_defaulter)   AS anc_defaulters,
    SUM(anc_complete)       AS anc_4plus,
    SUM(has_danger_signs)   AS danger_cases
  FROM chw_catalog.gold.fact_pregnancy GROUP BY ALL
),
pnc AS (
  SELECT date_key, chw_uuid,
    COUNT(*)                AS pnc_visits,
    SUM(pnc_upto_date)      AS pnc_current
  FROM chw_catalog.gold.fact_pnc GROUP BY ALL
),
fp AS (
  SELECT date_key, chw_uuid,
    COUNT(DISTINCT patient_id) AS fp_clients,
    SUM(is_on_fp)              AS on_fp
  FROM chw_catalog.gold.fact_family_planning GROUP BY ALL
),
pop AS (
  SELECT chw_uuid,
    SUM(u2_pop) AS u2, SUM(u5_pop) AS u5, SUM(wra_pop) AS wra
  FROM chw_catalog.gold.fact_population GROUP BY chw_uuid
)
SELECT
  d.month_label, d.year, d.quarter_label,
  c.county_name, c.sub_county_name, c.community_unit,
  c.chw_name,
  COALESCE(i.iz_visits, 0)            AS iz_visits,
  ROUND(100.0 * COALESCE(i.iz_fully_immunized, 0)
    / NULLIF(i.iz_visits, 0), 1)      AS immunization_coverage_pct,
  COALESCE(p.preg_registrations, 0)   AS preg_registrations,
  ROUND(100.0 * COALESCE(p.anc_defaulters, 0)
    / NULLIF(p.preg_registrations, 0), 1) AS anc_defaulter_pct,
  ROUND(100.0 * COALESCE(p.anc_4plus, 0)
    / NULLIF(p.preg_registrations, 0), 1) AS anc_4plus_pct,
  COALESCE(p.danger_cases, 0)         AS danger_sign_cases,
  COALESCE(n.pnc_visits, 0)           AS pnc_visits,
  ROUND(100.0 * COALESCE(n.pnc_current, 0)
    / NULLIF(n.pnc_visits, 0), 1)     AS pnc_coverage_pct,
  COALESCE(f.fp_clients, 0)           AS fp_clients,
  ROUND(100.0 * COALESCE(f.on_fp, 0)
    / NULLIF(po.wra, 0), 1)           AS mcpr_pct,
  COALESCE(po.u2, 0)                  AS pop_u2,
  COALESCE(po.u5, 0)                  AS pop_u5,
  COALESCE(po.wra, 0)                 AS pop_wra
FROM chw_catalog.gold.dim_date d
JOIN chw_catalog.gold.dim_chw c
LEFT JOIN imm i  ON d.date_key = i.date_key AND c.chw_uuid = i.chw_uuid
LEFT JOIN pnc n  ON d.date_key = n.date_key AND c.chw_uuid = n.chw_uuid
LEFT JOIN fp  f  ON d.date_key = f.date_key AND c.chw_uuid = f.chw_uuid
LEFT JOIN pop po ON c.chw_uuid = po.chw_uuid
LEFT JOIN preg p ON d.date_key = p.date_key
WHERE i.iz_visits IS NOT NULL
   OR p.preg_registrations IS NOT NULL
   OR n.pnc_visits IS NOT NULL
''')
print('✓ gold.vw_executive_summary')

# Coverage gap view
spark.sql('''
CREATE OR REPLACE VIEW chw_catalog.gold.vw_coverage_gaps AS
SELECT
  d.month_label, d.year,
  c.county_name, c.sub_county_name, c.community_unit,
  c.chw_uuid, c.chw_name,
  COALESCE(p.u2_pop, 0)                                       AS u2_pop,
  COALESCE(p.u5_pop, 0)                                       AS u5_pop,
  COALESCE(p.wra_pop, 0)                                      AS wra_pop,
  COALESCE(iz.iz_visits, 0)                                   AS iz_visits,
  COALESCE(p.u5_pop, 0) - COALESCE(iz.iz_visits, 0)          AS iz_unserved,
  ROUND(100.0 * COALESCE(iz.iz_visits, 0)
    / NULLIF(p.u5_pop, 0), 1)                                 AS iz_coverage_pct,
  COALESCE(pr.preg_count, 0)                                  AS preg_registered,
  ROUND(100.0 * COALESCE(fp.on_fp, 0)
    / NULLIF(p.wra_pop, 0), 1)                                AS mcpr_pct
FROM chw_catalog.gold.dim_chw c
CROSS JOIN chw_catalog.gold.dim_date d
LEFT JOIN (
  SELECT chw_uuid, SUM(u2_pop) AS u2_pop,
    SUM(u5_pop) AS u5_pop, SUM(wra_pop) AS wra_pop
  FROM chw_catalog.gold.fact_population
  GROUP BY chw_uuid
) p ON c.chw_uuid = p.chw_uuid
LEFT JOIN (
  SELECT date_key, chw_uuid, COUNT(*) AS iz_visits
  FROM chw_catalog.gold.fact_immunization GROUP BY ALL
) iz ON d.date_key = iz.date_key AND c.chw_uuid = iz.chw_uuid
LEFT JOIN (
  SELECT date_key, chu_code, COUNT(*) AS preg_count
  FROM chw_catalog.gold.fact_pregnancy GROUP BY ALL
) pr ON d.date_key = pr.date_key
LEFT JOIN (
  SELECT date_key, chw_uuid, SUM(is_on_fp) AS on_fp
  FROM chw_catalog.gold.fact_family_planning GROUP BY ALL
) fp ON d.date_key = fp.date_key AND c.chw_uuid = fp.chw_uuid
WHERE iz.iz_visits IS NOT NULL
   OR fp.on_fp IS NOT NULL
   OR pr.preg_count IS NOT NULL
''')
print('✓ gold.vw_coverage_gaps')

# CHW performance view
spark.sql('''
CREATE OR REPLACE VIEW chw_catalog.gold.vw_chw_performance AS
SELECT
  d.month_label, d.year,
  s.chw_uuid, c.chw_name, c.community_unit,
  c.county_name, c.sub_county_name,
  s.overall_score_pct,
  s.immunization_score_pct, s.pregnancy_score_pct,
  s.nutrition_score_pct,    s.malaria_score_pct,
  s.fp_score_pct,           s.newborn_score_pct,
  s.wash_score_pct,
  s.has_all_tools, s.has_ppe,
  COALESCE(hv.total_home_visits, 0) AS total_home_visits,
  CASE
    WHEN s.overall_score_pct >= 80 THEN "High performer"
    WHEN s.overall_score_pct >= 60 THEN "Meeting expectations"
    ELSE "Needs support" END        AS performance_tier
FROM chw_catalog.gold.fact_supervision s
JOIN chw_catalog.gold.dim_date d ON s.date_key = d.date_key
JOIN chw_catalog.gold.dim_chw  c ON s.chw_uuid = c.chw_uuid
LEFT JOIN (
  SELECT date_key, chw_uuid, SUM(visit_count) AS total_home_visits
  FROM chw_catalog.gold.fact_home_visit GROUP BY ALL
) hv ON s.date_key = hv.date_key AND s.chw_uuid = hv.chw_uuid
''')
print('✓ gold.vw_chw_performance')
print('\n✓ All semantic views created.')

✓ gold.vw_executive_summary
✓ gold.vw_coverage_gaps
✓ gold.vw_chw_performance

✓ All semantic views created.


In [0]:
# ── 04.2  Row-Level Security on all semantic views (Final Fixed Version) ─────

print("🔐 Applying county-based Row-Level Security to all semantic views...")

# 1. Executive Summary with RLS
spark.sql('''
CREATE OR REPLACE VIEW chw_catalog.gold.vw_executive_summary AS
WITH imm AS (
  SELECT date_key, chw_uuid,
    COUNT(*)                AS iz_visits,
    SUM(is_fully_immunized) AS iz_fully_immunized,
    SUM(is_defaulter)       AS iz_defaulters
  FROM chw_catalog.gold.fact_immunization GROUP BY ALL
),
preg AS (
  SELECT date_key, chu_code,
    COUNT(*)                AS preg_registrations,
    SUM(is_anc_defaulter)   AS anc_defaulters,
    SUM(anc_complete)       AS anc_4plus,
    SUM(has_danger_signs)   AS danger_cases
  FROM chw_catalog.gold.fact_pregnancy GROUP BY ALL
),
pnc AS (
  SELECT date_key, chw_uuid,
    COUNT(*)                AS pnc_visits,
    SUM(pnc_upto_date)      AS pnc_current
  FROM chw_catalog.gold.fact_pnc GROUP BY ALL
),
fp AS (
  SELECT date_key, chw_uuid,
    COUNT(DISTINCT patient_id) AS fp_clients,
    SUM(is_on_fp)              AS on_fp
  FROM chw_catalog.gold.fact_family_planning GROUP BY ALL
),
pop AS (
  SELECT chw_uuid,
    SUM(u2_pop) AS u2, SUM(u5_pop) AS u5, SUM(wra_pop) AS wra
  FROM chw_catalog.gold.fact_population GROUP BY chw_uuid
)
SELECT
  d.month_label, d.year, d.quarter_label,
  c.county_name, c.sub_county_name, c.community_unit,
  c.chw_name,
  COALESCE(i.iz_visits, 0)            AS iz_visits,
  ROUND(100.0 * COALESCE(i.iz_fully_immunized, 0)
    / NULLIF(i.iz_visits, 0), 1)      AS immunization_coverage_pct,
  COALESCE(p.preg_registrations, 0)   AS preg_registrations,
  ROUND(100.0 * COALESCE(p.anc_defaulters, 0)
    / NULLIF(p.preg_registrations, 0), 1) AS anc_defaulter_pct,
  ROUND(100.0 * COALESCE(p.anc_4plus, 0)
    / NULLIF(p.preg_registrations, 0), 1) AS anc_4plus_pct,
  COALESCE(p.danger_cases, 0)         AS danger_sign_cases,
  COALESCE(n.pnc_visits, 0)           AS pnc_visits,
  ROUND(100.0 * COALESCE(n.pnc_current, 0)
    / NULLIF(n.pnc_visits, 0), 1)     AS pnc_coverage_pct,
  COALESCE(f.fp_clients, 0)           AS fp_clients,
  ROUND(100.0 * COALESCE(f.on_fp, 0)
    / NULLIF(po.wra, 0), 1)           AS mcpr_pct,
  COALESCE(po.u2, 0)                  AS pop_u2,
  COALESCE(po.u5, 0)                  AS pop_u5,
  COALESCE(po.wra, 0)                 AS pop_wra
FROM chw_catalog.gold.dim_date d
JOIN chw_catalog.gold.dim_chw c
LEFT JOIN imm i  ON d.date_key = i.date_key AND c.chw_uuid = i.chw_uuid
LEFT JOIN pnc n  ON d.date_key = n.date_key AND c.chw_uuid = n.chw_uuid
LEFT JOIN fp  f  ON d.date_key = f.date_key AND c.chw_uuid = f.chw_uuid
LEFT JOIN pop po ON c.chw_uuid = po.chw_uuid
LEFT JOIN preg p ON d.date_key = p.date_key
WHERE 
  (i.iz_visits IS NOT NULL OR p.preg_registrations IS NOT NULL OR n.pnc_visits IS NOT NULL)
  AND (
    is_account_group_member("admins")
    OR c.county_name = REGEXP_EXTRACT(current_user(), "@([^_]+)", 1)
  )
''')
print('✓ vw_executive_summary with RLS')

# 2. Coverage Gaps with RLS
spark.sql('''
CREATE OR REPLACE VIEW chw_catalog.gold.vw_coverage_gaps AS
SELECT
  d.month_label, d.year,
  c.county_name, c.sub_county_name, c.community_unit,
  c.chw_uuid, c.chw_name,
  COALESCE(p.u2_pop, 0)                                       AS u2_pop,
  COALESCE(p.u5_pop, 0)                                       AS u5_pop,
  COALESCE(p.wra_pop, 0)                                      AS wra_pop,
  COALESCE(iz.iz_visits, 0)                                   AS iz_visits,
  COALESCE(p.u5_pop, 0) - COALESCE(iz.iz_visits, 0)          AS iz_unserved,
  ROUND(100.0 * COALESCE(iz.iz_visits, 0)
    / NULLIF(p.u5_pop, 0), 1)                                 AS iz_coverage_pct,
  COALESCE(pr.preg_count, 0)                                  AS preg_registered,
  ROUND(100.0 * COALESCE(fp.on_fp, 0)
    / NULLIF(p.wra_pop, 0), 1)                                AS mcpr_pct
FROM chw_catalog.gold.dim_chw c
CROSS JOIN chw_catalog.gold.dim_date d
LEFT JOIN (
  SELECT chw_uuid, SUM(u2_pop) AS u2_pop,
    SUM(u5_pop) AS u5_pop, SUM(wra_pop) AS wra_pop
  FROM chw_catalog.gold.fact_population
  GROUP BY chw_uuid
) p ON c.chw_uuid = p.chw_uuid
LEFT JOIN (
  SELECT date_key, chw_uuid, COUNT(*) AS iz_visits
  FROM chw_catalog.gold.fact_immunization GROUP BY ALL
) iz ON d.date_key = iz.date_key AND c.chw_uuid = iz.chw_uuid
LEFT JOIN (
  SELECT date_key, chu_code, COUNT(*) AS preg_count
  FROM chw_catalog.gold.fact_pregnancy GROUP BY ALL
) pr ON d.date_key = pr.date_key
LEFT JOIN (
  SELECT date_key, chw_uuid, SUM(is_on_fp) AS on_fp
  FROM chw_catalog.gold.fact_family_planning GROUP BY ALL
) fp ON d.date_key = fp.date_key AND c.chw_uuid = fp.chw_uuid
WHERE 
  (iz.iz_visits IS NOT NULL OR fp.on_fp IS NOT NULL OR pr.preg_count IS NOT NULL)
  AND (
    is_account_group_member("admins")
    OR c.county_name = REGEXP_EXTRACT(current_user(), "@([^_]+)", 1)
  )
''')
print('✓ vw_coverage_gaps with RLS')

# 3. CHW Performance with RLS
spark.sql('''
CREATE OR REPLACE VIEW chw_catalog.gold.vw_chw_performance AS
SELECT
  d.month_label, d.year,
  s.chw_uuid, c.chw_name, c.community_unit,
  c.county_name, c.sub_county_name,
  s.overall_score_pct,
  s.immunization_score_pct, s.pregnancy_score_pct,
  s.nutrition_score_pct,    s.malaria_score_pct,
  s.fp_score_pct,           s.newborn_score_pct,
  s.wash_score_pct,
  s.has_all_tools, s.has_ppe,
  COALESCE(hv.total_home_visits, 0) AS total_home_visits,
  CASE
    WHEN s.overall_score_pct >= 80 THEN "High performer"
    WHEN s.overall_score_pct >= 60 THEN "Meeting expectations"
    ELSE "Needs support" END        AS performance_tier
FROM chw_catalog.gold.fact_supervision s
JOIN chw_catalog.gold.dim_date d ON s.date_key = d.date_key
JOIN chw_catalog.gold.dim_chw  c ON s.chw_uuid = c.chw_uuid
LEFT JOIN (
  SELECT date_key, chw_uuid, SUM(visit_count) AS total_home_visits
  FROM chw_catalog.gold.fact_home_visit GROUP BY ALL
) hv ON s.date_key = hv.date_key AND s.chw_uuid = hv.chw_uuid
WHERE 
  is_account_group_member("admins")
  OR c.county_name = REGEXP_EXTRACT(current_user(), "@([^_]+)", 1)
''')
print('✓ vw_chw_performance with RLS')

# 4. Immunization fact RLS view
spark.sql('''
CREATE OR REPLACE VIEW chw_catalog.gold.vw_immunization_rls AS
SELECT f.*
FROM chw_catalog.gold.fact_immunization f
JOIN chw_catalog.gold.dim_chw c ON f.chw_uuid = c.chw_uuid
WHERE 
  is_account_group_member("admins") 
  OR c.county_name = REGEXP_EXTRACT(current_user(), "@([^_]+)", 1)
''')
print('✓ vw_immunization_rls')

print('\n✅ All semantic views now have county-based Row-Level Security.')

# ── GRANTS ───────────────────────────────────────────────────────────────
print("\nGranting SELECT permission to bi_users group...")

views = ['vw_executive_summary', 'vw_coverage_gaps', 'vw_chw_performance', 'vw_immunization_rls']

for view in views:
    try:
        spark.sql(f'GRANT SELECT ON VIEW chw_catalog.gold.{view} TO `bi_users`')
        print(f'  ✓ Granted SELECT on {view}')
    except Exception as e:
        err = str(e).lower()
        if "principal_does_not_exist" in err or "could not find principal" in err:
            print(f'  ⚠  Group `bi_users` not found. Please create it in Databricks Account Console → User management → Groups')
        else:
            print(f'  ✗ Error on {view}: {e}')

print('\n✅ Section 04.2 completed.')

🔐 Applying county-based Row-Level Security to all semantic views...
✓ vw_executive_summary with RLS
✓ vw_coverage_gaps with RLS
✓ vw_chw_performance with RLS
✓ vw_immunization_rls

✅ All semantic views now have county-based Row-Level Security.

Granting SELECT permission to bi_users group...
  ✓ Granted SELECT on vw_executive_summary
  ✓ Granted SELECT on vw_coverage_gaps
  ✓ Granted SELECT on vw_chw_performance
  ✓ Granted SELECT on vw_immunization_rls

✅ Section 04.2 completed.


In [0]:
# ── 04.2  Row-Level Security on all semantic views (county-based) ─────────────

print("Creating Row-Level Security on semantic views...")

# 1. Executive Summary View with RLS
spark.sql('''
CREATE OR REPLACE VIEW chw_catalog.gold.vw_executive_summary AS
WITH imm AS (
  SELECT date_key, chw_uuid,
    COUNT(*)                AS iz_visits,
    SUM(is_fully_immunized) AS iz_fully_immunized,
    SUM(is_defaulter)       AS iz_defaulters
  FROM chw_catalog.gold.fact_immunization GROUP BY ALL
),
preg AS (
  SELECT date_key, chu_code,
    COUNT(*)                AS preg_registrations,
    SUM(is_anc_defaulter)   AS anc_defaulters,
    SUM(anc_complete)       AS anc_4plus,
    SUM(has_danger_signs)   AS danger_cases
  FROM chw_catalog.gold.fact_pregnancy GROUP BY ALL
),
pnc AS (
  SELECT date_key, chw_uuid,
    COUNT(*)                AS pnc_visits,
    SUM(pnc_upto_date)      AS pnc_current
  FROM chw_catalog.gold.fact_pnc GROUP BY ALL
),
fp AS (
  SELECT date_key, chw_uuid,
    COUNT(DISTINCT patient_id) AS fp_clients,
    SUM(is_on_fp)              AS on_fp
  FROM chw_catalog.gold.fact_family_planning GROUP BY ALL
),
pop AS (
  SELECT chw_uuid,
    SUM(u2_pop) AS u2, SUM(u5_pop) AS u5, SUM(wra_pop) AS wra
  FROM chw_catalog.gold.fact_population GROUP BY chw_uuid
)
SELECT
  d.month_label, d.year, d.quarter_label,
  c.county_name, c.sub_county_name, c.community_unit,
  c.chw_name,
  COALESCE(i.iz_visits, 0)            AS iz_visits,
  ROUND(100.0 * COALESCE(i.iz_fully_immunized, 0)
    / NULLIF(i.iz_visits, 0), 1)      AS immunization_coverage_pct,
  COALESCE(p.preg_registrations, 0)   AS preg_registrations,
  ROUND(100.0 * COALESCE(p.anc_defaulters, 0)
    / NULLIF(p.preg_registrations, 0), 1) AS anc_defaulter_pct,
  ROUND(100.0 * COALESCE(p.anc_4plus, 0)
    / NULLIF(p.preg_registrations, 0), 1) AS anc_4plus_pct,
  COALESCE(p.danger_cases, 0)         AS danger_sign_cases,
  COALESCE(n.pnc_visits, 0)           AS pnc_visits,
  ROUND(100.0 * COALESCE(n.pnc_current, 0)
    / NULLIF(n.pnc_visits, 0), 1)     AS pnc_coverage_pct,
  COALESCE(f.fp_clients, 0)           AS fp_clients,
  ROUND(100.0 * COALESCE(f.on_fp, 0)
    / NULLIF(po.wra, 0), 1)           AS mcpr_pct,
  COALESCE(po.u2, 0)                  AS pop_u2,
  COALESCE(po.u5, 0)                  AS pop_u5,
  COALESCE(po.wra, 0)                 AS pop_wra
FROM chw_catalog.gold.dim_date d
JOIN chw_catalog.gold.dim_chw c
LEFT JOIN imm i  ON d.date_key = i.date_key AND c.chw_uuid = i.chw_uuid
LEFT JOIN pnc n  ON d.date_key = n.date_key AND c.chw_uuid = n.chw_uuid
LEFT JOIN fp  f  ON d.date_key = f.date_key AND c.chw_uuid = f.chw_uuid
LEFT JOIN pop po ON c.chw_uuid = po.chw_uuid
LEFT JOIN preg p ON d.date_key = p.date_key
WHERE 
  (i.iz_visits IS NOT NULL OR p.preg_registrations IS NOT NULL OR n.pnc_visits IS NOT NULL)
  AND (
    is_account_group_member("admins")
    OR c.county_name = REGEXP_EXTRACT(current_user(), "@([^_]+)", 1)
  )
''')
print('✓ vw_executive_summary with RLS')

# 2. Coverage Gaps View with RLS
spark.sql('''
CREATE OR REPLACE VIEW chw_catalog.gold.vw_coverage_gaps AS
SELECT
  d.month_label, d.year,
  c.county_name, c.sub_county_name, c.community_unit,
  c.chw_uuid, c.chw_name,
  COALESCE(p.u2_pop, 0)                                       AS u2_pop,
  COALESCE(p.u5_pop, 0)                                       AS u5_pop,
  COALESCE(p.wra_pop, 0)                                      AS wra_pop,
  COALESCE(iz.iz_visits, 0)                                   AS iz_visits,
  COALESCE(p.u5_pop, 0) - COALESCE(iz.iz_visits, 0)          AS iz_unserved,
  ROUND(100.0 * COALESCE(iz.iz_visits, 0)
    / NULLIF(p.u5_pop, 0), 1)                                 AS iz_coverage_pct,
  COALESCE(pr.preg_count, 0)                                  AS preg_registered,
  ROUND(100.0 * COALESCE(fp.on_fp, 0)
    / NULLIF(p.wra_pop, 0), 1)                                AS mcpr_pct
FROM chw_catalog.gold.dim_chw c
CROSS JOIN chw_catalog.gold.dim_date d
LEFT JOIN (
  SELECT chw_uuid, SUM(u2_pop) AS u2_pop,
    SUM(u5_pop) AS u5_pop, SUM(wra_pop) AS wra_pop
  FROM chw_catalog.gold.fact_population
  GROUP BY chw_uuid
) p ON c.chw_uuid = p.chw_uuid
LEFT JOIN (
  SELECT date_key, chw_uuid, COUNT(*) AS iz_visits
  FROM chw_catalog.gold.fact_immunization GROUP BY ALL
) iz ON d.date_key = iz.date_key AND c.chw_uuid = iz.chw_uuid
LEFT JOIN (
  SELECT date_key, chu_code, COUNT(*) AS preg_count
  FROM chw_catalog.gold.fact_pregnancy GROUP BY ALL
) pr ON d.date_key = pr.date_key
LEFT JOIN (
  SELECT date_key, chw_uuid, SUM(is_on_fp) AS on_fp
  FROM chw_catalog.gold.fact_family_planning GROUP BY ALL
) fp ON d.date_key = fp.date_key AND c.chw_uuid = fp.chw_uuid
WHERE 
  (iz.iz_visits IS NOT NULL OR fp.on_fp IS NOT NULL OR pr.preg_count IS NOT NULL)
  AND (
    is_account_group_member("admins")
    OR c.county_name = REGEXP_EXTRACT(current_user(), "@([^_]+)", 1)
  )
''')
print('✓ vw_coverage_gaps with RLS')

# 3. CHW Performance View with RLS
spark.sql('''
CREATE OR REPLACE VIEW chw_catalog.gold.vw_chw_performance AS
SELECT
  d.month_label, d.year,
  s.chw_uuid, c.chw_name, c.community_unit,
  c.county_name, c.sub_county_name,
  s.overall_score_pct,
  s.immunization_score_pct, s.pregnancy_score_pct,
  s.nutrition_score_pct,    s.malaria_score_pct,
  s.fp_score_pct,           s.newborn_score_pct,
  s.wash_score_pct,
  s.has_all_tools, s.has_ppe,
  COALESCE(hv.total_home_visits, 0) AS total_home_visits,
  CASE
    WHEN s.overall_score_pct >= 80 THEN "High performer"
    WHEN s.overall_score_pct >= 60 THEN "Meeting expectations"
    ELSE "Needs support" END        AS performance_tier
FROM chw_catalog.gold.fact_supervision s
JOIN chw_catalog.gold.dim_date d ON s.date_key = d.date_key
JOIN chw_catalog.gold.dim_chw  c ON s.chw_uuid = c.chw_uuid
LEFT JOIN (
  SELECT date_key, chw_uuid, SUM(visit_count) AS total_home_visits
  FROM chw_catalog.gold.fact_home_visit GROUP BY ALL
) hv ON s.date_key = hv.date_key AND s.chw_uuid = hv.chw_uuid
WHERE 
  is_account_group_member("admins")
  OR c.county_name = REGEXP_EXTRACT(current_user(), "@([^_]+)", 1)
''')
print('✓ vw_chw_performance with RLS')

# 4. (Optional) Keep a simple fact-level RLS view if needed for custom queries
spark.sql('''
CREATE OR REPLACE VIEW chw_catalog.gold.vw_immunization_rls AS
SELECT f.*
FROM chw_catalog.gold.fact_immunization f
JOIN chw_catalog.gold.dim_chw c ON f.chw_uuid = c.chw_uuid
WHERE 
  is_account_group_member("admins") 
  OR c.county_name = REGEXP_EXTRACT(current_user(), "@([^_]+)", 1)
''')
print('✓ vw_immunization_rls (fact-level)')

print('\n✓ All semantic views now have county-based Row-Level Security built-in.')

Creating Row-Level Security on semantic views...
✓ vw_executive_summary with RLS
✓ vw_coverage_gaps with RLS
✓ vw_chw_performance with RLS
✓ vw_immunization_rls (fact-level)

✓ All semantic views now have county-based Row-Level Security built-in.


In [0]:
# Ensure group exists and grant SELECT
try:
    spark.sql("CREATE GROUP bi_users")
    print('✓ Account group "bi_users" created')
except Exception as e:
    if "already exists" in str(e).lower():
        print('✓ Account group "bi_users" already exists')
    else:
        print(f"Group creation issue: {e}")

# Grant SELECT to bi_users on all views
for view in ['vw_executive_summary', 'vw_coverage_gaps', 
             'vw_chw_performance', 'vw_immunization_rls']:
    spark.sql(f"GRANT SELECT ON VIEW chw_catalog.gold.{view} TO bi_users")
    print(f"  ✓ Granted SELECT on {view} to bi_users")

print('\n✓ RLS setup and permissions completed.')

✓ Account group "bi_users" already exists
  ✓ Granted SELECT on vw_executive_summary to bi_users
  ✓ Granted SELECT on vw_coverage_gaps to bi_users
  ✓ Granted SELECT on vw_chw_performance to bi_users
  ✓ Granted SELECT on vw_immunization_rls to bi_users

✓ RLS setup and permissions completed.


## 04b · Unity Catalog Metric Views

> **What is a Metric View?**  
> A `CREATE METRIC VIEW` is a Unity Catalog first-class object (Databricks 14.3+).  
> It defines a **base table**, one or more **dimensions** (GROUP BY columns), and  
> named **measures** (aggregation expressions). Databricks AI/BI Genie discovers  
> these automatically and Power BI can query them as virtual tables.  
> Unlike plain views, metric views enforce a semantic contract — the same metric  
> definition is reused everywhere, preventing conflicting calculations across reports.

**Syntax:**
```sql
CREATE METRIC VIEW catalog.schema.metric_view_name
  DEFAULT METRICS (measure1, measure2, ...)
  DIMENSIONS (dim_col1, dim_col2, ...)
AS SELECT ...
```


In [0]:
# ── 04b  Unity Catalog Metric Views (Corrected YAML Syntax) ───────────────
print("Creating Unity Catalog Metric Views (YAML inside VIEW)...")

# 1. Immunization Metric View
spark.sql('''
CREATE OR REPLACE VIEW chw_catalog.gold.mv_immunization
WITH METRICS LANGUAGE YAML AS $$
version: 1.1
comment: "Immunization KPIs for CHW program"

source: |
  SELECT 
    fi.county,
    fi.reportedm,
    fi.chw_uuid,
    CASE
      WHEN fi.age_months < 12  THEN "0-11 months"
      WHEN fi.age_months < 24  THEN "12-23 months"
      WHEN fi.age_months < 60  THEN "24-59 months"
      ELSE "60+ months" END                       AS age_band,
    p.sex,

    COUNT(*)                                       AS immunization_visits,
    SUM(fi.is_fully_immunized)                     AS fully_immunized_count,
    ROUND(100.0 * SUM(fi.is_fully_immunized)
      / NULLIF(COUNT(*), 0), 2)                    AS immunization_coverage_pct,
    SUM(fi.is_defaulter)                           AS defaulter_count,
    ROUND(100.0 * SUM(fi.is_defaulter)
      / NULLIF(COUNT(*), 0), 2)                    AS defaulter_rate_pct,
    SUM(fi.penta1)                                 AS penta1_count,
    SUM(fi.penta3)                                 AS penta3_count,
    ROUND(100.0 * (SUM(fi.penta1) - SUM(fi.penta3))
      / NULLIF(SUM(fi.penta1), 0), 2)              AS penta_dropout_rate_pct,
    SUM(fi.bcg)                                    AS bcg_count,
    SUM(fi.measles_9m)                             AS measles_9m_count,
    SUM(fi.vitamin_a)                              AS vitamin_a_count,
    SUM(fi.is_referred)                            AS referral_count

  FROM chw_catalog.gold.fact_immunization fi
  LEFT JOIN chw_catalog.gold.dim_patient p USING (patient_id)

dimensions:
  - name: county
  - name: reportedm
  - name: chw_uuid
  - name: age_band
  - name: sex

measures:
  - name: immunization_visits          expression: immunization_visits
  - name: fully_immunized_count        expression: fully_immunized_count
  - name: immunization_coverage_pct    expression: immunization_coverage_pct
  - name: defaulter_count              expression: defaulter_count
  - name: defaulter_rate_pct           expression: defaulter_rate_pct
  - name: penta1_count                 expression: penta1_count
  - name: penta3_count                 expression: penta3_count
  - name: penta_dropout_rate_pct       expression: penta_dropout_rate_pct
  - name: bcg_count                    expression: bcg_count
  - name: measles_9m_count             expression: measles_9m_count
  - name: vitamin_a_count              expression: vitamin_a_count
  - name: referral_count               expression: referral_count
$$
''')
print('✓ mv_immunization')

Creating Unity Catalog Metric Views (YAML inside VIEW)...


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8777067039688815>, line 5
      2 print("Creating Unity Catalog Metric Views (YAML inside VIEW)...")
      4 # 1. Immunization Metric View
----> 5 spark.sql('''
      6 CREATE OR REPLACE VIEW chw_catalog.gold.mv_immunization
      7 WITH METRICS LANGUAGE YAML AS $$
      8 version: 1.1
      9 comment: "Immunization KPIs for CHW program"
     10 
     11 source: |
     12   SELECT 
     13     fi.county,
     14     fi.reportedm,
     15     fi.chw_uuid,
     16     CASE
     17       WHEN fi.age_months < 12  THEN "0-11 months"
     18       WHEN fi.age_months < 24  THEN "12-23 months"
     19       WHEN fi.age_months < 60  THEN "24-59 months"
     20       ELSE "60+ months" END                       AS age_band,
     21     p.sex,
     22 
     23     COUNT(*)                                       AS immunization_visits,


In [0]:
# ── 04b  Unity Catalog Metric Views ───────────────────────────────────────
# Requires Databricks Runtime 14.3+ and Unity Catalog enabled workspace.
# These are governance-enforced, Genie-discoverable semantic metrics.

# ── 1. Immunization Metric View ───────────────────────────────────────────
spark.sql('''
CREATE OR REPLACE METRIC VIEW chw_catalog.gold.mv_immunization
  DEFAULT METRICS (
    immunization_visits,
    fully_immunized_count,
    immunization_coverage_pct,
    defaulter_count,
    defaulter_rate_pct,
    penta1_count,
    penta3_count,
    penta_dropout_rate_pct,
    bcg_count,
    measles_9m_count,
    vitamin_a_count,
    referral_count
  )
  DIMENSIONS (
    county,
    reportedm,
    chw_uuid,
    age_band,
    sex
  )
AS
SELECT
  fi.county,
  fi.reportedm,
  fi.chw_uuid,
  CASE
    WHEN fi.age_months < 12  THEN "0-11 months"
    WHEN fi.age_months < 24  THEN "12-23 months"
    WHEN fi.age_months < 60  THEN "24-59 months"
    ELSE "60+ months" END                       AS age_band,
  p.sex,

  -- measures
  COUNT(*)                                       AS immunization_visits,
  SUM(fi.is_fully_immunized)                     AS fully_immunized_count,
  ROUND(100.0 * SUM(fi.is_fully_immunized)
    / NULLIF(COUNT(*), 0), 2)                    AS immunization_coverage_pct,
  SUM(fi.is_defaulter)                           AS defaulter_count,
  ROUND(100.0 * SUM(fi.is_defaulter)
    / NULLIF(COUNT(*), 0), 2)                    AS defaulter_rate_pct,
  SUM(fi.penta1)                                 AS penta1_count,
  SUM(fi.penta3)                                 AS penta3_count,
  ROUND(100.0 * (SUM(fi.penta1) - SUM(fi.penta3))
    / NULLIF(SUM(fi.penta1), 0), 2)              AS penta_dropout_rate_pct,
  SUM(fi.bcg)                                    AS bcg_count,
  SUM(fi.measles_9m)                             AS measles_9m_count,
  SUM(fi.vitamin_a)                              AS vitamin_a_count,
  SUM(fi.is_referred)                            AS referral_count

FROM chw_catalog.gold.fact_immunization fi
LEFT JOIN chw_catalog.gold.dim_patient p USING (patient_id)
GROUP BY ALL
''')
print('✓ mv_immunization')


# ── 2. Maternal Health Metric View ────────────────────────────────────────
spark.sql('''
CREATE OR REPLACE METRIC VIEW chw_catalog.gold.mv_maternal_health
  DEFAULT METRICS (
    pregnancy_registrations,
    anc_started_count,
    anc_4plus_count,
    anc_4plus_completion_pct,
    anc_defaulter_count,
    anc_defaulter_rate_pct,
    danger_sign_count,
    danger_sign_rate_pct,
    on_iron_folate_count,
    iron_folate_coverage_pct,
    muac_red_count,
    muac_yellow_count,
    referral_count
  )
  DIMENSIONS (
    county,
    reportedm,
    muac_risk,
    source_form
  )
AS
SELECT
  fp.county,
  fp.reportedm,
  fp.muac_risk,
  fp.source_form,

  COUNT(*)                                        AS pregnancy_registrations,
  SUM(fp.has_started_anc)                         AS anc_started_count,
  SUM(fp.anc_complete)                            AS anc_4plus_count,
  ROUND(100.0 * SUM(fp.anc_complete)
    / NULLIF(COUNT(*), 0), 2)                     AS anc_4plus_completion_pct,
  SUM(fp.is_anc_defaulter)                        AS anc_defaulter_count,
  ROUND(100.0 * SUM(fp.is_anc_defaulter)
    / NULLIF(COUNT(*), 0), 2)                     AS anc_defaulter_rate_pct,
  SUM(fp.has_danger_signs)                        AS danger_sign_count,
  ROUND(100.0 * SUM(fp.has_danger_signs)
    / NULLIF(COUNT(*), 0), 2)                     AS danger_sign_rate_pct,
  SUM(fp.on_iron_folate)                          AS on_iron_folate_count,
  ROUND(100.0 * SUM(fp.on_iron_folate)
    / NULLIF(COUNT(*), 0), 2)                     AS iron_folate_coverage_pct,
  SUM(CASE WHEN fp.muac_color = "red" THEN 1 ELSE 0 END)    AS muac_red_count,
  SUM(CASE WHEN fp.muac_color = "yellow" THEN 1 ELSE 0 END)  AS muac_yellow_count,
  SUM(fp.is_referred)                             AS referral_count

FROM chw_catalog.gold.fact_pregnancy fp
GROUP BY ALL
''')
print('✓ mv_maternal_health')


# ── 3. Family Planning Metric View ────────────────────────────────────────
spark.sql('''
CREATE OR REPLACE METRIC VIEW chw_catalog.gold.mv_family_planning
  DEFAULT METRICS (
    fp_clients,
    on_fp_count,
    mcpr_pct,
    refill_count,
    refill_rate_pct,
    side_effect_count,
    side_effect_rate_pct,
    coc_cycles_dispensed,
    pop_cycles_dispensed,
    condoms_dispensed
  )
  DIMENSIONS (
    county,
    reportedm,
    chw_uuid,
    fp_method,
    fp_method_category,
    sex,
    age_band
  )
AS
SELECT
  f.county,
  f.reportedm,
  f.chw_uuid,
  f.fp_method,
  f.fp_method_category,
  f.sex,
  CASE
    WHEN f.age_years < 20 THEN "15-19"
    WHEN f.age_years < 25 THEN "20-24"
    WHEN f.age_years < 30 THEN "25-29"
    WHEN f.age_years < 35 THEN "30-34"
    WHEN f.age_years < 50 THEN "35-49"
    ELSE "50+" END                              AS age_band,

  COUNT(DISTINCT f.patient_id)                  AS fp_clients,
  SUM(f.is_on_fp)                               AS on_fp_count,
  -- mCPR requires wra_pop denominator — joined below
  ROUND(100.0 * SUM(f.is_on_fp)
    / NULLIF(SUM(p.wra_pop), 0), 2)             AS mcpr_pct,
  SUM(f.refilled_today)                         AS refill_count,
  ROUND(100.0 * SUM(f.refilled_today)
    / NULLIF(COUNT(*), 0), 2)                   AS refill_rate_pct,
  SUM(f.has_side_effects)                       AS side_effect_count,
  ROUND(100.0 * SUM(f.has_side_effects)
    / NULLIF(COUNT(*), 0), 2)                   AS side_effect_rate_pct,
  SUM(f.coc_cycles)                             AS coc_cycles_dispensed,
  SUM(f.pop_cycles)                             AS pop_cycles_dispensed,
  SUM(f.condom_pieces)                          AS condoms_dispensed

FROM chw_catalog.gold.fact_family_planning f
LEFT JOIN (
  SELECT chw_uuid, reportedm AS report_month, SUM(wra_pop) AS wra_pop
  FROM chw_catalog.gold.fact_population GROUP BY ALL
) p ON f.chw_uuid = p.chw_uuid AND f.reportedm = p.report_month
GROUP BY ALL
''')
print('✓ mv_family_planning')


# ── 4. Postnatal Care Metric View ─────────────────────────────────────────
spark.sql('''
CREATE OR REPLACE METRIC VIEW chw_catalog.gold.mv_postnatal_care
  DEFAULT METRICS (
    pnc_visits,
    pnc_upto_date_count,
    pnc_coverage_pct,
    danger_followup_count,
    maternal_death_count,
    newborn_home_visits,
    fp_initiation_count,
    fp_initiation_rate_pct,
    facility_delivery_count,
    facility_delivery_rate_pct
  )
  DIMENSIONS (
    county,
    reportedm,
    chw_uuid,
    place_of_delivery
  )
AS
SELECT
  pnc.county,
  pnc.reportedm,
  pnc.chw_uuid,
  pnc.place_of_delivery,

  COUNT(*)                                        AS pnc_visits,
  SUM(pnc.pnc_upto_date)                          AS pnc_upto_date_count,
  ROUND(100.0 * SUM(pnc.pnc_upto_date)
    / NULLIF(COUNT(*), 0), 2)                     AS pnc_coverage_pct,
  SUM(pnc.needs_danger_followup)                  AS danger_followup_count,
  SUM(pnc.is_maternal_death)                      AS maternal_death_count,
  SUM(pnc.newborn_visits)                         AS newborn_home_visits,
  SUM(pnc.started_fp)                             AS fp_initiation_count,
  ROUND(100.0 * SUM(pnc.started_fp)
    / NULLIF(COUNT(*), 0), 2)                     AS fp_initiation_rate_pct,
  SUM(CASE WHEN pnc.place_of_delivery = "facility"
    THEN 1 ELSE 0 END)                            AS facility_delivery_count,
  ROUND(100.0 * SUM(CASE WHEN pnc.place_of_delivery = "facility"
    THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0), 2) AS facility_delivery_rate_pct

FROM chw_catalog.gold.fact_pnc pnc
GROUP BY ALL
''')
print('✓ mv_postnatal_care')


# ── 5. CHW Supervision Metric View ────────────────────────────────────────
spark.sql('''
CREATE OR REPLACE METRIC VIEW chw_catalog.gold.mv_supervision
  DEFAULT METRICS (
    supervision_visits,
    avg_overall_score_pct,
    avg_immunization_score_pct,
    avg_pregnancy_score_pct,
    avg_nutrition_score_pct,
    avg_malaria_score_pct,
    avg_fp_score_pct,
    avg_newborn_score_pct,
    avg_wash_score_pct,
    tools_readiness_pct,
    ppe_readiness_pct,
    high_performer_count,
    needs_support_count
  )
  DIMENSIONS (
    county,
    reportedm,
    chw_uuid,
    performance_tier
  )
AS
SELECT
  s.county,
  s.reportedm,
  s.chw_uuid,
  CASE
    WHEN s.overall_score_pct >= 80 THEN "High performer"
    WHEN s.overall_score_pct >= 60 THEN "Meeting expectations"
    ELSE "Needs support" END                      AS performance_tier,

  COUNT(*)                                        AS supervision_visits,
  ROUND(AVG(s.overall_score_pct), 2)              AS avg_overall_score_pct,
  ROUND(AVG(s.immunization_score_pct), 2)         AS avg_immunization_score_pct,
  ROUND(AVG(s.pregnancy_score_pct), 2)            AS avg_pregnancy_score_pct,
  ROUND(AVG(s.nutrition_score_pct), 2)            AS avg_nutrition_score_pct,
  ROUND(AVG(s.malaria_score_pct), 2)              AS avg_malaria_score_pct,
  ROUND(AVG(s.fp_score_pct), 2)                   AS avg_fp_score_pct,
  ROUND(AVG(s.newborn_score_pct), 2)              AS avg_newborn_score_pct,
  ROUND(AVG(s.wash_score_pct), 2)                 AS avg_wash_score_pct,
  ROUND(100.0 * SUM(s.has_all_tools)
    / NULLIF(COUNT(*), 0), 2)                     AS tools_readiness_pct,
  ROUND(100.0 * SUM(s.has_ppe)
    / NULLIF(COUNT(*), 0), 2)                     AS ppe_readiness_pct,
  SUM(CASE WHEN s.overall_score_pct >= 80 THEN 1 ELSE 0 END) AS high_performer_count,
  SUM(CASE WHEN s.overall_score_pct <  60 THEN 1 ELSE 0 END) AS needs_support_count

FROM chw_catalog.gold.fact_supervision s
GROUP BY ALL
''')
print('✓ mv_supervision')


# ── 6. Coverage Gap Metric View (denominator-driven) ──────────────────────
spark.sql('''
CREATE OR REPLACE VIEW chw_catalog.gold.mv_coverage_gaps
  DEFAULT METRICS (
    u2_population,
    u5_population,
    wra_population,
    iz_visits,
    iz_coverage_rate_pct,
    iz_unserved_children,
    preg_registrations,
    fp_clients_on_fp,
    mcpr_rate_pct,
    fp_unserved_wra
  )
  DIMENSIONS (
    county,
    report_month,
    chw_uuid,
    sub_county_name,
    community_unit
  )
AS
SELECT
  p.county,
  p.report_month,
  p.chw_uuid,
  c.sub_county_name,
  c.community_unit,

  SUM(p.u2_pop)                                   AS u2_population,
  SUM(p.u5_pop)                                   AS u5_population,
  SUM(p.wra_pop)                                  AS wra_population,

  COALESCE(SUM(iz.iz_visits), 0)                  AS iz_visits,
  ROUND(100.0 * COALESCE(SUM(iz.iz_visits), 0)
    / NULLIF(SUM(p.u5_pop), 0), 2)                AS iz_coverage_rate_pct,
  SUM(p.u5_pop) - COALESCE(SUM(iz.iz_visits), 0) AS iz_unserved_children,

  COALESCE(SUM(pr.preg_count), 0)                 AS preg_registrations,

  COALESCE(SUM(fp.on_fp), 0)                      AS fp_clients_on_fp,
  ROUND(100.0 * COALESCE(SUM(fp.on_fp), 0)
    / NULLIF(SUM(p.wra_pop), 0), 2)               AS mcpr_rate_pct,
  SUM(p.wra_pop) - COALESCE(SUM(fp.on_fp), 0)    AS fp_unserved_wra

FROM chw_catalog.gold.fact_population p
JOIN chw_catalog.gold.dim_chw c USING (chw_uuid)
LEFT JOIN (
  SELECT
    chw_uuid,
    CAST(DATE_FORMAT(DATE_TRUNC("MONTH",
      TO_DATE(reportedm, "MMMyyyyy")), "yyyyMM") AS STRING) AS rpt_month,
    COUNT(*) AS iz_visits
  FROM chw_catalog.gold.fact_immunization GROUP BY ALL
) iz ON p.chw_uuid = iz.chw_uuid AND p.report_month = iz.rpt_month
LEFT JOIN (
  SELECT county,
    reportedm, COUNT(*) AS preg_count
  FROM chw_catalog.gold.fact_pregnancy GROUP BY ALL
) pr ON p.county = pr.county AND p.report_month = pr.reportedm
LEFT JOIN (
  SELECT chw_uuid, reportedm, SUM(is_on_fp) AS on_fp
  FROM chw_catalog.gold.fact_family_planning GROUP BY ALL
) fp ON p.chw_uuid = fp.chw_uuid AND p.report_month = fp.reportedm
GROUP BY ALL
''')
print('✓ mv_coverage_gaps')

print('\n✓ All 6 Metric Views created.')
print('  These are discoverable by Databricks AI/BI Genie and'
      ' queryable via Power BI DirectQuery.')

In [0]:
# ── Grant SELECT on all metric views to BI consumers ──────────────────────

metric_views = [
    'mv_immunization',
    'mv_maternal_health',
    'mv_family_planning',
    'mv_postnatal_care',
    'mv_supervision',
    'mv_coverage_gaps',
]

for mv in metric_views:
    spark.sql(f'GRANT SELECT ON chw_catalog.gold.{mv} TO `bi_users`')
    spark.sql(f'GRANT SELECT ON chw_catalog.gold.{mv} TO `genie_users`')
    print(f'  ✓ Grants applied: {mv}')

# Verify all metric views exist
print('\nMetric views in chw_catalog.gold:')
spark.sql("""
    SHOW TABLES IN chw_catalog.gold
""").filter("tableName LIKE 'mv_%'").show(truncate=False)

### Metric View Summary

| Metric View | Measures | Key Dimensions | Fact Table |
|---|---|---|---|
| `mv_immunization` | 12 | county, reportedm, chw_uuid, age_band, sex | fact_immunization |
| `mv_maternal_health` | 13 | county, reportedm, muac_risk, source_form | fact_pregnancy |
| `mv_family_planning` | 10 | county, reportedm, fp_method, fp_method_category, age_band | fact_family_planning |
| `mv_postnatal_care` | 10 | county, reportedm, chw_uuid, place_of_delivery | fact_pnc |
| `mv_supervision` | 13 | county, reportedm, chw_uuid, performance_tier | fact_supervision |
| `mv_coverage_gaps` | 10 | county, report_month, chw_uuid, community_unit | fact_population |

**How AI/BI Genie uses these:**  
Once registered, Genie can answer natural-language questions like:  
- *"Which county has the lowest immunization coverage this month?"*  
- *"Show me CHWs with supervision score below 60 in Busia"*  
- *"What is the mCPR trend over the last 4 months?"*

**Power BI:**  
Metric views appear as virtual tables in the Databricks connector.  
Query them exactly like regular Delta tables — all measures and dimensions  
are pre-defined columns you can drag into visuals.

## 05 · Data Quality Validation

In [0]:
# ── 05.1  Row counts per layer ─────────────────────────────────────────────

layers = {
    'bronze': ['iz','supervision','preg_reg','preg_reg2',
               'preg_visit','preg_visit2','pnc','fp',
               'refill','homevisit','population','active_chps'],
    'silver': ['immunization','pregnancy_registration',
               'postnatal_care','family_planning','supervision',
               'population','home_visit','active_chps'],
    'gold':   ['dim_chw','dim_date','dim_geography','dim_facility','dim_patient',
               'fact_immunization','fact_pregnancy','fact_pnc',
               'fact_family_planning','fact_supervision',
               'fact_population','fact_home_visit'],
}

print(f'{"Layer":<8} {"Table":<35} {"Rows":>10}')
print('-' * 58)
for layer, tables in layers.items():
    for t in tables:
        try:
            n = spark.sql(
                f'SELECT COUNT(*) AS n FROM chw_catalog.{layer}.{t}'
            ).first()['n']
            print(f'{layer:<8} {t:<35} {n:>10,}')
        except Exception as e:
            print(f'{layer:<8} {t:<35} {"ERROR":>10} — {e}')

In [0]:
# ── 05.2  KPI sanity checks ───────────────────────────────────────────────

spark.sql('''
SELECT metric, ROUND(value, 1) AS value, expected_range FROM (
  SELECT "immunization_coverage_pct" AS metric,
    100.0 * SUM(is_fully_immunized) / COUNT(*) AS value,
    "30-95%" AS expected_range
  FROM chw_catalog.gold.fact_immunization
  UNION ALL
  SELECT "anc_defaulter_pct",
    100.0 * SUM(is_anc_defaulter) / COUNT(*),
    "5-40%"
  FROM chw_catalog.gold.fact_pregnancy
  UNION ALL
  SELECT "penta_dropout_pct",
    100.0 * (SUM(penta1) - SUM(penta3)) / NULLIF(SUM(penta1), 0),
    "0-40%"
  FROM chw_catalog.gold.fact_immunization
  UNION ALL
  SELECT "avg_supervision_score_pct",
    AVG(overall_score_pct),
    "40-90%"
  FROM chw_catalog.gold.fact_supervision
  UNION ALL
  SELECT "pnc_upto_date_pct",
    100.0 * SUM(pnc_upto_date) / COUNT(*),
    "30-80%"
  FROM chw_catalog.gold.fact_pnc
)
''').show(truncate=False)

## 06 · Power BI Connection Guide & DAX Measures

### Step 1 — Get SQL Warehouse connection details
In your Databricks workspace: **Compute → SQL Warehouses → your warehouse → Connection details**
Copy the **Server hostname** and **HTTP path**.

### Step 2 — Connect Power BI Desktop
1. Open Power BI Desktop
2. **Get Data → Databricks** (not Azure Databricks — the plain Databricks connector)
3. Paste `Server hostname` and `HTTP path`
4. Set data connectivity mode: **DirectQuery**
5. Authenticate with **OAuth2** or Personal Access Token

### Step 3 — Import tables
Navigate to `chw_catalog → gold` and import:
- `vw_executive_summary` (pre-aggregated, fastest)
- `vw_coverage_gaps`
- `vw_chw_performance`
- `fact_immunization`, `fact_pregnancy`, `fact_pnc`, `fact_family_planning`, `fact_supervision`
- `dim_chw`, `dim_date`, `dim_geography`

### Step 4 — Set relationships in Model view
```
fact_immunization[chw_uuid]       → dim_chw[chw_uuid]    (Many:1, Active)
fact_immunization[date_key]       → dim_date[date_key]   (Many:1, Active)
fact_pregnancy[chw_uuid]          → dim_chw[chw_uuid]    (Many:1, Inactive)
fact_pnc[chw_uuid]                → dim_chw[chw_uuid]    (Many:1, Inactive)
fact_family_planning[chw_uuid]    → dim_chw[chw_uuid]    (Many:1, Inactive)
fact_supervision[chw_uuid]        → dim_chw[chw_uuid]    (Many:1, Inactive)
```
Use `USERELATIONSHIP()` in DAX to activate inactive relationships per measure.

### Step 5 — DAX Measures

**Immunization**
```dax
Immunization Coverage % =
DIVIDE(SUM(fact_immunization[is_fully_immunized]), COUNTROWS(fact_immunization)) * 100

Penta Dropout Rate % =
DIVIDE(
    SUM(fact_immunization[penta1]) - SUM(fact_immunization[penta3]),
    SUM(fact_immunization[penta1])
) * 100

Immunization Gap =
SUM(fact_population[u5_pop]) - COUNTROWS(fact_immunization)
```

**Maternal health**
```dax
ANC Defaulter Rate % =
DIVIDE(SUM(fact_pregnancy[is_anc_defaulter]), COUNTROWS(fact_pregnancy)) * 100

ANC 4+ Completion % =
DIVIDE(SUM(fact_pregnancy[anc_complete]), COUNTROWS(fact_pregnancy)) * 100

Danger Sign Prevalence % =
DIVIDE(SUM(fact_pregnancy[has_danger_signs]), COUNTROWS(fact_pregnancy)) * 100
```

**Family planning**
```dax
mCPR % =
DIVIDE(
    CALCULATE(SUM(fact_family_planning[is_on_fp])),
    CALCULATE(SUM(fact_population[wra_pop]))
) * 100

FP Refill Rate % =
DIVIDE(SUM(fact_family_planning[refilled_today]),
       COUNTROWS(fact_family_planning)) * 100
```

**CHW performance**
```dax
Avg Supervision Score =
AVERAGE(fact_supervision[overall_score_pct])

CHW Productivity Index =
DIVIDE(
    COUNTROWS(fact_immunization) + COUNTROWS(fact_pregnancy),
    DISTINCTCOUNT(fact_immunization[chw_uuid])
)
```


## 07 · Pregnancy Journey — Longitudinal Episode Table

> **Why this matters:**  
> Every other table in this project is *event-level* — one row per visit, per registration, per dose.  
> `fact_pregnancy_journey` is *episode-level* — one row per woman's entire pregnancy, stitching  
> together registration → ANC visits → PNC into a single longitudinal record.  
> This enables **funnel analysis**, **dropout attribution**, and **cohort tracking** —  
> the three analytical patterns that separate health data science from health data reporting.

### Episode funnel
```
Stage 1: Registered           ← preg_reg / preg_reg2
Stage 2: ANC initiated        ← has_started_anc = yes
Stage 3: ANC 1-3 visits       ← preg_visit / preg_visit2
Stage 4: ANC 4+ complete      ← WHO minimum standard
Stage 5: PNC reached          ← pnc
Stage 6: FP post-delivery     ← fp / refill
```

### Key derived fields
| Field | Definition |
|---|---|
| `funnel_stage_reached` | Ordinal 1–6, highest stage attained |
| `dropout_stage` | Label for the stage where the woman was lost |
| `pnc_within_48hrs` | Global health standard: PNC contact ≤ 2 days post-delivery |
| `facility_delivery` | Place of delivery = facility |
| `had_danger_signs` | Any danger sign recorded at any visit |
| `days_registration_to_first_anc` | Timeliness of first ANC contact |
| `days_delivery_to_pnc` | Timeliness of postnatal care |


In [0]:
# ── 07.1  silver.pregnancy_visit (prerequisite — merges preg_visit + preg_visit2)
# This was defined earlier but not written to Silver. Creating it now.

spark.sql('''
CREATE OR REPLACE TABLE chw_catalog.silver.pregnancy_visit AS
WITH base AS (
  SELECT
    uuid,
    CAST(reported AS TIMESTAMP)                              AS reported_at,
    county, month, reportedm,
    patient_id, patient_name,
    CAST(patient_age_in_years AS INT)                        AS age_years,
    CAST(COALESCE(has_started_anc = "yes", false) AS INT)    AS has_started_anc,
    COALESCE(CAST(number_of_anc_attended AS INT), 0)         AS anc_visits,
    CAST(next_anc_visit_date AS DATE)                        AS next_anc_date,
    CAST(current_edd AS DATE)                                AS edd,
    period_of_pregnancy,
    CAST(COALESCE(has_danger_signs = "yes", false) AS INT)   AS has_danger_signs,
    muac_color,
    CAST(COALESCE(takes_iron_or_folate_supplements = "yes", false) AS INT) AS on_iron_folate,
    CAST(COALESCE(CAST(is_anc_defaulter AS BOOLEAN), false) AS INT) AS is_anc_defaulter,
    CAST(COALESCE(has_been_referred = "yes", false) AS INT)  AS is_referred,
    chu_code, chu_name,
    referred_to_facility_code, referred_to_facility_name,
    "preg_visit" AS source_form
  FROM chw_catalog.bronze.preg_visit WHERE uuid IS NOT NULL

  UNION ALL

  SELECT
    uuid, CAST(reported AS TIMESTAMP),
    county, month, reportedm, patient_id, patient_name,
    CAST(patient_age_in_years AS INT),
    CAST(COALESCE(has_started_anc = "yes", false) AS INT),
    COALESCE(CAST(number_of_anc_attended AS INT), 0),
    CAST(next_anc_visit_date AS DATE),
    CAST(current_edd AS DATE),
    period_of_pregnancy,
    CAST(COALESCE(has_danger_signs = "yes", false) AS INT),
    muac_color,
    CAST(COALESCE(takes_iron_or_folate_supplements = "yes", false) AS INT),
    CAST(COALESCE(CAST(is_anc_defaulter AS BOOLEAN), false) AS INT),
    CAST(COALESCE(has_been_referred = "yes", false) AS INT),
    chu_code, chu_name,
    referred_to_facility_code, referred_to_facility_name,
    "preg_visit2" AS source_form
  FROM chw_catalog.bronze.preg_visit2 WHERE uuid IS NOT NULL
)
SELECT *, current_timestamp() AS silver_loaded_at FROM base
QUALIFY ROW_NUMBER() OVER (PARTITION BY uuid ORDER BY reported_at DESC) = 1
''')
print("✓ silver.pregnancy_visit")

In [0]:
# ── 07.2  gold.fact_pregnancy_journey ─────────────────────────────────────
# One row per woman per pregnancy episode.
# Stitches: registration → ANC visits → PNC → FP post-delivery.

spark.sql('''
CREATE OR REPLACE TABLE chw_catalog.gold.fact_pregnancy_journey AS

WITH registration AS (
  -- ── Base episode: one row per registered pregnancy ────────────────────
  SELECT
    uuid                                           AS registration_uuid,
    patient_id,
    county,
    reportedm,
    reported_at                                    AS registered_at,
    age_years,
    edd,
    muac_color,
    has_started_anc,
    anc_visits                                     AS anc_visits_at_registration,
    has_danger_signs                               AS danger_signs_at_registration,
    on_iron_folate                                 AS iron_folate_at_registration,
    is_anc_defaulter                               AS defaulter_at_registration,
    is_referred                                    AS referred_at_registration,
    chu_code, chu_name,
    source_form
  FROM chw_catalog.silver.pregnancy_registration
),

visit_summary AS (
  -- ── ANC visit history aggregated per patient ──────────────────────────
  SELECT
    patient_id,
    COUNT(*)                                       AS visit_records_count,
    MAX(anc_visits)                                AS max_anc_visits_reported,
    MIN(reported_at)                               AS first_visit_date,
    MAX(reported_at)                               AS last_visit_date,
    MAX(has_danger_signs)                          AS danger_signs_any_visit,
    MAX(on_iron_folate)                            AS iron_folate_any_visit,
    MAX(is_anc_defaulter)                          AS ever_defaulter,
    MAX(is_referred)                               AS ever_referred_from_visit,
    -- most severe MUAC across all visits
    CASE
      WHEN MAX(CASE WHEN muac_color = "red"    THEN 3 ELSE 0 END) = 3 THEN "red"
      WHEN MAX(CASE WHEN muac_color = "yellow" THEN 2 ELSE 0 END) = 2 THEN "yellow"
      ELSE "green" END                            AS worst_muac_across_visits
  FROM chw_catalog.silver.pregnancy_visit
  GROUP BY patient_id
),

pnc_summary AS (
  -- ── First PNC contact per patient ────────────────────────────────────
  SELECT
    patient_id,
    MIN(reported_at)                               AS first_pnc_date,
    MIN(delivery_date)                             AS delivery_date,
    MIN(days_since_delivery)                       AS days_delivery_to_first_pnc,
    MAX(pnc_visit_count)                           AS total_pnc_visits,
    MAX(pnc_upto_date)                             AS pnc_upto_date,
    MAX(started_fp)                                AS fp_initiated_at_pnc,
    MAX(is_maternal_death)                         AS is_maternal_death,
    MAX(needs_danger_followup)                     AS pnc_danger_followup,
    -- facility delivery
    MAX(CASE WHEN LOWER(place_of_delivery)
      LIKE "%facility%"
      OR LOWER(place_of_delivery)
      LIKE "%hospital%"
      OR LOWER(place_of_delivery)
      LIKE "%clinic%"
      THEN 1 ELSE 0 END)                          AS facility_delivery
  FROM chw_catalog.silver.postnatal_care
  GROUP BY patient_id
),

fp_post AS (
  -- ── FP uptake any time (captures post-delivery FP initiation) ─────────
  SELECT
    patient_id,
    MAX(is_on_fp)                                  AS on_fp,
    MAX(fp_method)                                 AS fp_method_chosen,
    MIN(CASE WHEN is_on_fp = 1
        THEN reported_at END)                      AS fp_start_date
  FROM chw_catalog.silver.family_planning
  GROUP BY patient_id
),

joined AS (
  SELECT
    r.registration_uuid,
    r.patient_id,
    r.county,
    r.reportedm,
    r.registered_at,
    r.age_years,
    r.edd,
    r.chu_code, r.chu_name,
    r.source_form,

    -- ── MUAC risk (worst across all contacts) ─────────────────────────
    CASE
      WHEN r.muac_color = "red"
        OR COALESCE(v.worst_muac_across_visits,"") = "red"     THEN "red"
      WHEN r.muac_color = "yellow"
        OR COALESCE(v.worst_muac_across_visits,"") = "yellow"  THEN "yellow"
      ELSE "green" END                            AS muac_worst,

    -- ── ANC visit count (best available source) ───────────────────────
    GREATEST(
      COALESCE(v.max_anc_visits_reported, 0),
      COALESCE(r.anc_visits_at_registration, 0)
    )                                              AS total_anc_visits,

    -- ── Danger signs (any contact) ────────────────────────────────────
    GREATEST(
      COALESCE(r.danger_signs_at_registration, 0),
      COALESCE(v.danger_signs_any_visit, 0)
    )                                              AS had_danger_signs,

    -- ── Iron / folate (any contact) ───────────────────────────────────
    GREATEST(
      COALESCE(r.iron_folate_at_registration, 0),
      COALESCE(v.iron_folate_any_visit, 0)
    )                                              AS ever_on_iron_folate,

    -- ── PNC fields ────────────────────────────────────────────────────
    p.delivery_date,
    p.days_delivery_to_first_pnc,
    p.total_pnc_visits,
    COALESCE(p.pnc_upto_date, 0)                   AS pnc_upto_date,
    COALESCE(p.fp_initiated_at_pnc, 0)             AS fp_initiated_at_pnc,
    COALESCE(p.is_maternal_death, 0)               AS is_maternal_death,
    COALESCE(p.pnc_danger_followup, 0)             AS pnc_danger_followup,
    COALESCE(p.facility_delivery, 0)               AS facility_delivery,

    -- ── FP post-delivery ──────────────────────────────────────────────
    COALESCE(fp.on_fp, 0)                          AS on_fp_any_time,
    fp.fp_method_chosen,

    -- ── References ────────────────────────────────────────────────────
    v.first_visit_date,
    v.last_visit_date,
    v.ever_defaulter,
    COALESCE(v.ever_referred_from_visit,
             r.referred_at_registration, 0)        AS ever_referred

  FROM registration r
  LEFT JOIN visit_summary v ON r.patient_id = v.patient_id
  LEFT JOIN pnc_summary   p ON r.patient_id = p.patient_id
  LEFT JOIN fp_post       fp ON r.patient_id = fp.patient_id
)

SELECT
  registration_uuid,
  patient_id,
  county, reportedm,
  registered_at, age_years, edd,
  chu_code, chu_name, source_form,
  muac_worst,
  had_danger_signs,
  ever_on_iron_folate,
  ever_referred,
  total_anc_visits,

  -- ── Funnel stage flags (1/0 at each stage) ──────────────────────────
  1                                                AS stage_1_registered,
  CAST(COALESCE(had_danger_signs, 0) AS INT)       AS has_risk_factors,

  CASE WHEN total_anc_visits >= 1 THEN 1 ELSE 0 END AS stage_2_anc1,
  CASE WHEN total_anc_visits >= 2 THEN 1 ELSE 0 END AS stage_3_anc2,
  CASE WHEN total_anc_visits >= 3 THEN 1 ELSE 0 END AS stage_4_anc3,
  CASE WHEN total_anc_visits >= 4 THEN 1 ELSE 0 END AS stage_5_anc4_plus,
  CASE WHEN delivery_date IS NOT NULL
    OR total_pnc_visits IS NOT NULL THEN 1
    ELSE 0 END                                     AS stage_6_delivery_reached,
  CASE WHEN total_pnc_visits > 0 THEN 1 ELSE 0 END AS stage_7_pnc_reached,
  CASE WHEN on_fp_any_time = 1 THEN 1 ELSE 0 END  AS stage_8_fp_postpartum,

  -- ── Global health outcome flags ───────────────────────────────────────
  CASE WHEN COALESCE(days_delivery_to_first_pnc, 999) <= 2
    THEN 1 ELSE 0 END                              AS pnc_within_48hrs,
  facility_delivery,
  pnc_upto_date,
  fp_initiated_at_pnc,
  on_fp_any_time,
  fp_method_chosen,
  is_maternal_death,
  pnc_danger_followup,

  -- ── Timeliness metrics (days) ─────────────────────────────────────────
  DATEDIFF(first_visit_date, registered_at)        AS days_registration_to_first_anc,
  DATEDIFF(last_visit_date, first_visit_date)      AS days_span_anc_visits,
  COALESCE(days_delivery_to_first_pnc, -1)         AS days_delivery_to_pnc,

  -- ── Highest funnel stage reached (ordinal 1–8) ───────────────────────
  CASE
    WHEN on_fp_any_time = 1                         THEN 8
    WHEN total_pnc_visits > 0                       THEN 7
    WHEN delivery_date IS NOT NULL                  THEN 6
    WHEN total_anc_visits >= 4                      THEN 5
    WHEN total_anc_visits >= 3                      THEN 4
    WHEN total_anc_visits >= 2                      THEN 3
    WHEN total_anc_visits >= 1                      THEN 2
    ELSE 1 END                                      AS funnel_stage_reached,

  -- ── Dropout stage label ───────────────────────────────────────────────
  CASE
    WHEN on_fp_any_time = 1                         THEN "Completed full continuum"
    WHEN total_pnc_visits > 0                       THEN "Dropped after PNC"
    WHEN delivery_date IS NOT NULL                  THEN "Dropped before PNC"
    WHEN total_anc_visits >= 4                      THEN "Dropped after ANC4+"
    WHEN total_anc_visits >= 3                      THEN "Dropped after ANC3"
    WHEN total_anc_visits >= 2                      THEN "Dropped after ANC2"
    WHEN total_anc_visits >= 1                      THEN "Dropped after ANC1"
    ELSE "Registered only — never attended ANC" END AS dropout_stage,

  -- ── MUAC risk category ────────────────────────────────────────────────
  CASE muac_worst
    WHEN "red"    THEN "High risk"
    WHEN "yellow" THEN "Moderate risk"
    ELSE "Low risk" END                             AS muac_risk_category,

  -- ── Age group ─────────────────────────────────────────────────────────
  CASE
    WHEN age_years < 18               THEN "Adolescent (<18)"
    WHEN age_years BETWEEN 18 AND 24  THEN "Young adult (18-24)"
    WHEN age_years BETWEEN 25 AND 34  THEN "Prime reproductive (25-34)"
    WHEN age_years >= 35              THEN "Advanced maternal age (35+)"
    ELSE "Unknown" END                              AS maternal_age_group,

  current_timestamp()                              AS gold_loaded_at

FROM joined
''')

count = spark.sql("SELECT COUNT(*) AS n FROM chw_catalog.gold.fact_pregnancy_journey").first()["n"]
print(f"✓ gold.fact_pregnancy_journey — {count:,} episodes")

In [0]:
# ── 07.3  Funnel summary view — ready for Power BI funnel chart ───────────

spark.sql('''
CREATE OR REPLACE VIEW chw_catalog.gold.vw_maternal_funnel AS
SELECT
  county,
  reportedm,
  -- Total cohort at each funnel stage
  COUNT(*)                                         AS total_registered,
  SUM(stage_2_anc1)                                AS reached_anc1,
  SUM(stage_3_anc2)                                AS reached_anc2,
  SUM(stage_4_anc3)                                AS reached_anc3,
  SUM(stage_5_anc4_plus)                           AS reached_anc4_plus,
  SUM(stage_6_delivery_reached)                    AS reached_delivery,
  SUM(stage_7_pnc_reached)                         AS reached_pnc,
  SUM(stage_8_fp_postpartum)                       AS reached_fp_postpartum,

  -- Conversion rates (% of registered)
  ROUND(100.0 * SUM(stage_2_anc1)         / NULLIF(COUNT(*), 0), 1) AS pct_reached_anc1,
  ROUND(100.0 * SUM(stage_5_anc4_plus)    / NULLIF(COUNT(*), 0), 1) AS pct_anc4_complete,
  ROUND(100.0 * SUM(stage_7_pnc_reached)  / NULLIF(COUNT(*), 0), 1) AS pct_reached_pnc,
  ROUND(100.0 * SUM(stage_8_fp_postpartum)/ NULLIF(COUNT(*), 0), 1) AS pct_fp_postpartum,

  -- Dropout between consecutive stages (absolute count)
  COUNT(*) - SUM(stage_2_anc1)                     AS dropout_before_anc1,
  SUM(stage_2_anc1) - SUM(stage_3_anc2)            AS dropout_between_anc1_anc2,
  SUM(stage_3_anc2) - SUM(stage_4_anc3)            AS dropout_between_anc2_anc3,
  SUM(stage_4_anc3) - SUM(stage_5_anc4_plus)       AS dropout_between_anc3_anc4,
  SUM(stage_5_anc4_plus) - SUM(stage_7_pnc_reached) AS dropout_anc4_to_pnc,
  SUM(stage_7_pnc_reached) - SUM(stage_8_fp_postpartum) AS dropout_pnc_to_fp,

  -- Dropout rates between stages
  ROUND(100.0 * (SUM(stage_2_anc1) - SUM(stage_3_anc2))
    / NULLIF(SUM(stage_2_anc1), 0), 1)             AS dropout_rate_anc1_anc2,
  ROUND(100.0 * (SUM(stage_3_anc2) - SUM(stage_4_anc3))
    / NULLIF(SUM(stage_3_anc2), 0), 1)             AS dropout_rate_anc2_anc3,
  ROUND(100.0 * (SUM(stage_4_anc3) - SUM(stage_5_anc4_plus))
    / NULLIF(SUM(stage_4_anc3), 0), 1)             AS dropout_rate_anc3_anc4,

  -- Global health outcomes
  SUM(facility_delivery)                           AS facility_deliveries,
  SUM(pnc_within_48hrs)                            AS pnc_within_48hrs,
  SUM(had_danger_signs)                            AS total_with_danger_signs,
  SUM(ever_on_iron_folate)                         AS ever_on_iron_folate,
  SUM(is_maternal_death)                           AS maternal_deaths,
  ROUND(100.0 * SUM(facility_delivery)
    / NULLIF(SUM(stage_6_delivery_reached), 0), 1) AS facility_delivery_rate_pct,
  ROUND(100.0 * SUM(pnc_within_48hrs)
    / NULLIF(SUM(stage_7_pnc_reached), 0), 1)      AS pnc_48hr_rate_pct,

  -- Timeliness
  ROUND(AVG(CASE WHEN days_registration_to_first_anc >= 0
    THEN days_registration_to_first_anc END), 1)   AS avg_days_reg_to_anc1,
  ROUND(AVG(CASE WHEN days_delivery_to_pnc >= 0
    THEN days_delivery_to_pnc END), 1)             AS avg_days_delivery_to_pnc

FROM chw_catalog.gold.fact_pregnancy_journey
GROUP BY ALL
''')
print("✓ gold.vw_maternal_funnel")


# ── Dropout detail view — for drill-through in Power BI ───────────────────
spark.sql('''
CREATE OR REPLACE VIEW chw_catalog.gold.vw_dropout_analysis AS
SELECT
  county,
  reportedm,
  dropout_stage,
  muac_risk_category,
  maternal_age_group,
  COUNT(*)                                         AS women_count,
  SUM(had_danger_signs)                            AS with_danger_signs,
  SUM(ever_referred)                               AS ever_referred,
  ROUND(AVG(total_anc_visits), 1)                  AS avg_anc_visits,
  ROUND(AVG(age_years), 1)                         AS avg_age
FROM chw_catalog.gold.fact_pregnancy_journey
GROUP BY ALL
ORDER BY county, reportedm, funnel_stage_reached
''')
print("✓ gold.vw_dropout_analysis")

In [0]:
# ── 07.4  Metric View for pregnancy journey ────────────────────────────────

spark.sql('''
CREATE OR REPLACE METRIC VIEW chw_catalog.gold.mv_pregnancy_journey
  DEFAULT METRICS (
    total_episodes,
    anc1_count,
    anc4_complete_count,
    anc1_conversion_pct,
    anc4_completion_pct,
    pnc_reached_count,
    pnc_coverage_pct,
    pnc_within_48hrs_count,
    pnc_48hr_rate_pct,
    facility_delivery_count,
    facility_delivery_rate_pct,
    fp_postpartum_count,
    fp_postpartum_rate_pct,
    danger_sign_count,
    danger_sign_rate_pct,
    iron_folate_coverage_pct,
    maternal_death_count,
    avg_days_reg_to_anc1,
    avg_days_delivery_to_pnc,
    dropout_before_anc1,
    dropout_anc3_to_anc4
  )
  DIMENSIONS (
    county,
    reportedm,
    dropout_stage,
    muac_risk_category,
    maternal_age_group,
    facility_delivery
  )
AS
SELECT
  county,
  reportedm,
  dropout_stage,
  muac_risk_category,
  maternal_age_group,
  facility_delivery,

  COUNT(*)                                          AS total_episodes,
  SUM(stage_2_anc1)                                 AS anc1_count,
  SUM(stage_5_anc4_plus)                            AS anc4_complete_count,
  ROUND(100.0 * SUM(stage_2_anc1)
    / NULLIF(COUNT(*), 0), 2)                       AS anc1_conversion_pct,
  ROUND(100.0 * SUM(stage_5_anc4_plus)
    / NULLIF(COUNT(*), 0), 2)                       AS anc4_completion_pct,
  SUM(stage_7_pnc_reached)                          AS pnc_reached_count,
  ROUND(100.0 * SUM(stage_7_pnc_reached)
    / NULLIF(COUNT(*), 0), 2)                       AS pnc_coverage_pct,
  SUM(pnc_within_48hrs)                             AS pnc_within_48hrs_count,
  ROUND(100.0 * SUM(pnc_within_48hrs)
    / NULLIF(SUM(stage_7_pnc_reached), 0), 2)       AS pnc_48hr_rate_pct,
  SUM(facility_delivery)                            AS facility_delivery_count,
  ROUND(100.0 * SUM(facility_delivery)
    / NULLIF(SUM(stage_6_delivery_reached), 0), 2)  AS facility_delivery_rate_pct,
  SUM(stage_8_fp_postpartum)                        AS fp_postpartum_count,
  ROUND(100.0 * SUM(stage_8_fp_postpartum)
    / NULLIF(COUNT(*), 0), 2)                       AS fp_postpartum_rate_pct,
  SUM(had_danger_signs)                             AS danger_sign_count,
  ROUND(100.0 * SUM(had_danger_signs)
    / NULLIF(COUNT(*), 0), 2)                       AS danger_sign_rate_pct,
  ROUND(100.0 * SUM(ever_on_iron_folate)
    / NULLIF(COUNT(*), 0), 2)                       AS iron_folate_coverage_pct,
  SUM(is_maternal_death)                            AS maternal_death_count,
  ROUND(AVG(CASE WHEN days_registration_to_first_anc >= 0
    THEN days_registration_to_first_anc END), 1)    AS avg_days_reg_to_anc1,
  ROUND(AVG(CASE WHEN days_delivery_to_pnc >= 0
    THEN days_delivery_to_pnc END), 1)              AS avg_days_delivery_to_pnc,
  COUNT(*) - SUM(stage_2_anc1)                      AS dropout_before_anc1,
  SUM(stage_4_anc3) - SUM(stage_5_anc4_plus)        AS dropout_anc3_to_anc4

FROM chw_catalog.gold.fact_pregnancy_journey
GROUP BY ALL
''')
print("✓ mv_pregnancy_journey")

# Grant access
for obj in [
    "TABLE chw_catalog.gold.fact_pregnancy_journey",
    "VIEW chw_catalog.gold.vw_maternal_funnel",
    "VIEW chw_catalog.gold.vw_dropout_analysis",
    "chw_catalog.gold.mv_pregnancy_journey",
]:
    spark.sql(f"GRANT SELECT ON {obj} TO `bi_users`")
print("✓ Grants applied")

In [0]:
# ── 07.5  Validate + preview the funnel ───────────────────────────────────

print("=" * 60)
print("PREGNANCY JOURNEY — FUNNEL SUMMARY")
print("=" * 60)

spark.sql('''
SELECT
  COUNT(*)                                         AS total_pregnancies,
  SUM(stage_2_anc1)                                AS reached_anc1,
  SUM(stage_5_anc4_plus)                           AS completed_anc4plus,
  SUM(stage_7_pnc_reached)                         AS reached_pnc,
  SUM(pnc_within_48hrs)                            AS pnc_within_48hrs,
  SUM(facility_delivery)                           AS facility_deliveries,
  SUM(stage_8_fp_postpartum)                       AS on_fp_postpartum,
  SUM(had_danger_signs)                            AS had_danger_signs,
  SUM(is_maternal_death)                           AS maternal_deaths
FROM chw_catalog.gold.fact_pregnancy_journey
''').show(truncate=False)

print("\nDROPOUT BREAKDOWN BY STAGE:")
spark.sql('''
SELECT dropout_stage, COUNT(*) AS women, ROUND(100.0 * COUNT(*)
  / SUM(COUNT(*)) OVER (), 1) AS pct
FROM chw_catalog.gold.fact_pregnancy_journey
GROUP BY dropout_stage
ORDER BY COUNT(*) DESC
''').show(truncate=False)

print("\nBY COUNTY:")
spark.sql('''
SELECT county, total_registered, reached_anc1, reached_anc4_plus,
  pct_anc4_complete, pct_reached_pnc, pnc_48hr_rate_pct, facility_delivery_rate_pct
FROM chw_catalog.gold.vw_maternal_funnel
ORDER BY county, reportedm
''').show(truncate=False)

### Power BI — Maternal Funnel Page

**Tables to import:** `fact_pregnancy_journey`, `vw_maternal_funnel`, `vw_dropout_analysis`

**Funnel chart visual:**
- Source: `vw_maternal_funnel`
- Values field (in order): `total_registered → reached_anc1 → reached_anc2 → reached_anc3 → reached_anc4_plus → reached_pnc → reached_fp_postpartum`
- Slicers: county, reportedm, maternal_age_group

**DAX measures for this page:**
```dax
Biggest Dropout Stage =
TOPN(1,
    SUMMARIZE(vw_dropout_analysis, vw_dropout_analysis[dropout_stage],
              "n", SUM(vw_dropout_analysis[women_count])),
    [n], DESC
)[dropout_stage]

ANC3 to ANC4 Dropout % =
DIVIDE(
    SUM(fact_pregnancy_journey[stage_4_anc3])
    - SUM(fact_pregnancy_journey[stage_5_anc4_plus]),
    SUM(fact_pregnancy_journey[stage_4_anc3])
) * 100

PNC Within 48hrs % =
DIVIDE(
    SUM(fact_pregnancy_journey[pnc_within_48hrs]),
    SUM(fact_pregnancy_journey[stage_7_pnc_reached])
) * 100

Facility Delivery Rate % =
DIVIDE(
    SUM(fact_pregnancy_journey[facility_delivery]),
    SUM(fact_pregnancy_journey[stage_6_delivery_reached])
) * 100
```

**Interview talking point:**
> *"The biggest drop-off in this cohort occurs between ANC3 and ANC4, which typically maps to the third trimester when travel burden increases. This is exactly the intervention point a program manager would target — not more registrations, but retention between visit 3 and 4."*